# STABL baseline study-by-protection immune signature analysis

This notebook uses multinomial elastic-net STABL to identify baseline immune signatures that distinguish the crossed study/protection groups `EG_P`, `EG_NP`, `TU_P`, `TU_NP`, `GA_P`, and `GA_NP`. It uses the 11-view RaJIVE preprocessing scheme for sample alignment, missing-data handling, feature filtering, robust scaling, and block normalization, but it does not run RaJIVE, dream models, jackstraw, or component-score analyses.

P/NP status is kept as descriptive plot annotation only. The supervised target throughout this notebook is `study_group`, so any P/NP pattern should be interpreted as post hoc QC rather than a modeled contrast.


## Setup And Run Controls

This section defines the input paths, reproducibility seed, STABL settings, cache/export locations, and run flags. The default settings load cached branch outputs produced by `scratch/scripts/run_stablr_baseline_study_protection_branch.R` and the paired SLURM jobs; local refits are opt-in so readers can navigate the notebook without accidentally launching long jobs.


In [1]:
# Runtime controls -------------------------------------------------------------
ALL_VIEW_PATHS <- c(
  cytof_celltype = "/exports/para-lipg-hpc/Xuran/data/pilot/cytof/analysis/clus_freq_clr_combat.csv",
  exvivo_celltype = "/exports/para-lipg-hpc/Xuran/data/pilot/aurora_exvivo/analysis/clus_freq_combat.csv",
  exvivo_enzyme = "/exports/para-lipg-hpc/Xuran/data/pilot/aurora_exvivo/analysis/enzyme_celltype_combat.csv",
  `6h_cyto_LPS` = "/exports/para-lipg-hpc/Xuran/data/pilot/aurora_6H/analysis/cytokine_pos_6hLPS_combat.csv",
  `6h_cyto_ssRNA40` = "/exports/para-lipg-hpc/Xuran/data/pilot/aurora_6H/analysis/cytokine_pos_6hssRNA40_combat.csv",
  `24h_cyto_iRBC` = "/exports/para-lipg-hpc/Xuran/data/pilot/aurora_24H/analysis/cytokine_pos_24hiRBC_combat.csv",
  `24h_cyto_SEB` = "/exports/para-lipg-hpc/Xuran/data/pilot/aurora_24H/analysis/cytokine_pos_24hSEB_combat.csv",
  `24h_enzyme_iRBC` = "/exports/para-lipg-hpc/Xuran/data/pilot/aurora_24H/analysis/enzyme_celltype_24hiRBC_combat.csv",
  `24h_enzyme_SEB` = "/exports/para-lipg-hpc/Xuran/data/pilot/aurora_24H/analysis/enzyme_celltype_24hSEB_combat.csv",
  `3d_cyto` = "/exports/para-lipg-hpc/Xuran/data/pilot/aurora_3D/analysis/cytokine_pos_combat.csv",
  `3d_enzyme` = "/exports/para-lipg-hpc/Xuran/data/pilot/aurora_3D/analysis/enzyme_celltype_combat.csv"
)
CYTOF_CELLTYPE_PATH <- unname(ALL_VIEW_PATHS["cytof_celltype"])
METADATA_PATH <- "/exports/para-lipg-hpc/Xuran/data/metadata.csv"

STABLR_SEED <- 20260512L
IMPUTATION_SEED <- 101L
STABL_FAMILY <- "multinomial"
STABL_BASE_LEARNER <- "elastic_net"
ELASTIC_NET_L1_RATIO <- 0.5
N_BOOTSTRAPS <- 500L
N_LAMBDA <- 20L
SAMPLE_FRACTION <- 0.9  # Imbalanced six-label target; keeps tiny TU_NP represented in stratified resamples.
ARTIFICIAL_TYPE <- "random_permutation"
ARTIFICIAL_PROPORTION <- 1.0
FDR_THRESHOLD_RANGE <- seq(0, 0.99, by = 0.01)
N_WORKERS <- as.integer(Sys.getenv("SLURM_CPUS_PER_TASK", "8"))
N_ITER_LF <- 5000L
TOP_FALLBACK_FEATURES <- 5L
TOP_HEATMAP_FEATURES <- 50L
TOP_FEATURES_PER_GROUP <- 5L
COOPERATIVE_RHO <- c(0, 0.1, 0.25, 0.5, 1)

LOAD_CACHED_RESULTS <- TRUE     # Prefer caches produced by scratch/scripts and scratch/slurm jobs.
RUN_CYTOF_SANITY_CHECK <- FALSE       # Set TRUE to refit locally when the cached CyTOF branch is absent.
RUN_ALL_SINGLE_VIEW_STABL <- FALSE    # Set TRUE to refit all single-view branches locally.
RUN_ALL_VIEW_EARLY_FUSION <- FALSE    # Set TRUE to refit the early-fusion branch locally.
RUN_LATE_FUSION <- FALSE              # Heavy; normally load scratch/cache/.../late_fusion/late_fusion_result.rds.
RUN_OVR_STABL <- FALSE                # Heavy direct binomial one-vs-rest branches; normally load cached fits.
RUN_COOPERATIVE_OVR <- FALSE          # Heavy auxiliary branches; normally load cached one-vs-rest fits.
RUN_CLUSTERING_VISUALIZATIONS <- FALSE # Set TRUE to regenerate heatmaps/embeddings from the loaded early-fusion fit.
RUN_PUBLICATION_NESTED_CV <- FALSE    # Heavy validation; normally load scratch/cache/.../nested_cv/nested_cv_result.rds.
FORCE_RECOMPUTE <- FALSE
WRITE_NOTEBOOK_OUTPUTS <- FALSE # Set TRUE to write regenerated notebook tables and figures.
CACHE_DIR <- file.path("scratch", "cache", "stablr_baseline_study_protection_test")
EXPORT_DIR <- file.path("scratch", "outputs", "stablr_baseline_study_protection_test")

STUDY_GROUP_MAP <- c(CVTU3 = "TU", EGSV2 = "EG", PfGA2 = "GA")
STUDY_ARM_LEVELS <- c("EG", "GA", "TU")
STUDY_GROUP_LEVELS <- c("EG_P", "EG_NP", "TU_P", "TU_NP", "GA_P", "GA_NP")
STUDY_GROUP_COLORS <- c(EG_P = "#4C78A8", EG_NP = "#9EC5E6", TU_P = "#54A24B", TU_NP = "#A1D99B", GA_P = "#F58518", GA_NP = "#FFBF79")
EXPECTED_BASELINE_N <- 38L
EXPECTED_BASELINE_COUNTS <- c(EG_P = 6L, EG_NP = 4L, TU_P = 9L, TU_NP = 3L, GA_P = 8L, GA_NP = 8L)


In [2]:
# Package setup ----------------------------------------------------------------
find_stablr_pkg_root <- function(start = getwd()) {
  path <- normalizePath(start, winslash = "/", mustWork = TRUE)
  repeat {
    candidates <- c(file.path(path, "r-pkg", "stablr"), path)
    for (candidate in candidates) {
      desc <- file.path(candidate, "DESCRIPTION")
      if (file.exists(desc)) {
        fields <- tryCatch(read.dcf(desc), error = function(e) NULL)
        if (!is.null(fields) && "Package" %in% colnames(fields) &&
            identical(unname(fields[1, "Package"]), "stablr")) {
          return(candidate)
        }
      }
    }
    parent <- dirname(path)
    if (identical(parent, path)) break
    path <- parent
  }
  NA_character_
}

stablr_pkg_root <- find_stablr_pkg_root()
if (!is.na(stablr_pkg_root) && requireNamespace("devtools", quietly = TRUE)) {
  suppressPackageStartupMessages(devtools::load_all(stablr_pkg_root, quiet = TRUE))
} else {
  suppressPackageStartupMessages(library(stablr))
}

infer_stablr_repo_root <- function(pkg_root) {
  if (!is.na(pkg_root)) {
    repo_candidate <- normalizePath(file.path(pkg_root, "..", ".."), winslash = "/", mustWork = FALSE)
    if (file.exists(file.path(repo_candidate, "AGENTS.md")) && dir.exists(file.path(repo_candidate, "scratch"))) {
      return(repo_candidate)
    }
  }
  normalizePath(getwd(), winslash = "/", mustWork = TRUE)
}

stablr_repo_root <- infer_stablr_repo_root(stablr_pkg_root)
resolve_repo_relative_path <- function(path) {
  vapply(path, function(one_path) {
    if (is.na(one_path) || grepl("^(/|[A-Za-z]:[\\/])", one_path)) {
      return(one_path)
    }
    file.path(stablr_repo_root, one_path)
  }, character(1L), USE.NAMES = FALSE)
}

CACHE_DIR <- resolve_repo_relative_path(CACHE_DIR)
EXPORT_DIR <- resolve_repo_relative_path(EXPORT_DIR)

required_packages <- c("dplyr", "tidyr", "ggplot2", "tibble", "purrr", "readr", "forcats")
optional_packages <- c("missForest", "mixOmics", "glmnet", "ggpubr", "ComplexHeatmap", "pheatmap", "circlize", "uwot", "multiview")
missing_packages <- required_packages[!vapply(required_packages, requireNamespace, logical(1L), quietly = TRUE)]
if (length(missing_packages) > 0L) {
  stop("Install required packages before running this notebook: ",
       paste(missing_packages, collapse = ", "), call. = FALSE)
}
suppressPackageStartupMessages(invisible(lapply(required_packages, library, character.only = TRUE)))

missing_optional <- optional_packages[!vapply(optional_packages, requireNamespace, logical(1L), quietly = TRUE)]
if (length(missing_optional) > 0L) {
  message("Optional packages not available: ", paste(missing_optional, collapse = ", "),
          ". Sections that need them will stop with a targeted message.")
}

dir.create(CACHE_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(EXPORT_DIR, recursive = TRUE, showWarnings = FALSE)


# Artifact helpers -------------------------------------------------------------
ARTIFACT_MANIFEST <- tibble::tibble(
  branch = character(),
  object_type = character(),
  name = character(),
  path = character(),
  timestamp = character()
)

branch_cache_dir <- function(branch = "notebook") {
  path <- file.path(CACHE_DIR, branch)
  dir.create(path, recursive = TRUE, showWarnings = FALSE)
  path
}

branch_export_dir <- function(branch = "notebook", type = NULL) {
  path <- if (is.null(type)) file.path(EXPORT_DIR, branch) else file.path(EXPORT_DIR, branch, type)
  dir.create(path, recursive = TRUE, showWarnings = FALSE)
  path
}

cached_rds_path <- function(branch, name) {
  file.path(branch_cache_dir(branch), paste0(artifact_name(name), ".rds"))
}

first_existing_path <- function(paths) {
  hits <- unname(paths[file.exists(paths)])
  if (length(hits) == 0L) return(NA_character_)
  hits[[1L]]
}

load_cached_rds <- function(branch, name, required = FALSE) {
  path <- cached_rds_path(branch, name)
  if (file.exists(path)) {
    message("Loaded cached RDS: ", normalizePath(path, winslash = "/", mustWork = FALSE))
    return(readRDS(path))
  }
  if (isTRUE(required)) {
    stop("Missing cached RDS: ", path, call. = FALSE)
  }
  NULL
}

cached_table_path <- function(branch, name) {
  file.path(branch_export_dir(branch, "tables"), paste0(artifact_name(name), ".csv"))
}

load_cached_table <- function(branch, name) {
  path <- cached_table_path(branch, name)
  if (file.exists(path)) {
    message("Loaded cached table: ", normalizePath(path, winslash = "/", mustWork = FALSE))
    return(readr::read_csv(path, show_col_types = FALSE))
  }
  tibble::tibble()
}

cached_artifact_paths <- function(branch, type = "figures") {
  path <- branch_export_dir(branch, type)
  if (!dir.exists(path)) return(character())
  list.files(path, full.names = TRUE)
}

artifact_name <- function(filename) {
  tools::file_path_sans_ext(basename(filename))
}

infer_artifact_branch <- function(filename) {
  nm <- artifact_name(filename)
  dplyr::case_when(
    grepl("^cytof", nm) ~ "cytof",
    grepl("^single_view", nm) ~ "single_view",
    grepl("^late_fusion", nm) ~ "late_fusion",
    grepl("^cooperative", nm) ~ "cooperative_ovr",
    grepl("^nested_cv", nm) ~ "nested_cv",
    grepl("^all_view|^top_study|^cross_view|^immune_signature|^selected_feature|^feature_overlap", nm) ~ "early_fusion",
    grepl("^preprocess|^baseline|^input_file|^near_zero|^scaling|^original", nm) ~ "preprocess",
    TRUE ~ "notebook"
  )
}

flush_artifact_manifest <- function() {
  saveRDS(ARTIFACT_MANIFEST, file.path(CACHE_DIR, "artifact_manifest.rds"))
  readr::write_csv(ARTIFACT_MANIFEST, file.path(EXPORT_DIR, "artifact_manifest.csv"))
  invisible(ARTIFACT_MANIFEST)
}

record_artifact <- function(branch, object_type, name, path) {
  ARTIFACT_MANIFEST <<- dplyr::bind_rows(
    ARTIFACT_MANIFEST,
    tibble::tibble(
      branch = branch,
      object_type = object_type,
      name = name,
      path = normalizePath(path, winslash = "/", mustWork = FALSE),
      timestamp = format(Sys.time(), "%Y-%m-%d %H:%M:%S %Z")
    )
  ) |>
    dplyr::distinct()
  flush_artifact_manifest()
  invisible(path)
}

cache_object <- function(object, name, branch = "notebook") {
  path <- file.path(branch_cache_dir(branch), paste0(artifact_name(name), ".rds"))
  saveRDS(object, path)
  record_artifact(branch, "rds", artifact_name(name), path)
  invisible(path)
}

export_table <- function(x, name, branch = infer_artifact_branch(name)) {
  if (!isTRUE(WRITE_NOTEBOOK_OUTPUTS)) return(invisible(x))
  path <- file.path(branch_export_dir(branch, "tables"), paste0(artifact_name(name), ".csv"))
  readr::write_csv(x, path)
  record_artifact(branch, "csv", artifact_name(name), path)
  invisible(x)
}

export_plot <- function(p, name, branch = infer_artifact_branch(name), width = 10, height = 7) {
  if (!isTRUE(WRITE_NOTEBOOK_OUTPUTS)) return(invisible(p))
  fig_dir <- branch_export_dir(branch, "figures")
  png_path <- file.path(fig_dir, paste0(artifact_name(name), ".png"))
  pdf_path <- file.path(fig_dir, paste0(artifact_name(name), ".pdf"))
  ggplot2::ggsave(png_path, p, width = width, height = height, dpi = 300)
  ggplot2::ggsave(pdf_path, p, width = width, height = height)
  record_artifact(branch, "png", artifact_name(name), png_path)
  record_artifact(branch, "pdf", artifact_name(name), pdf_path)
  invisible(p)
}

RNGkind("L'Ecuyer-CMRG")
set.seed(STABLR_SEED)


Warning message:
“package ‘testthat’ was built under R version 4.5.2”
Warning message:
“package ‘dplyr’ was built under R version 4.5.2”
Warning message:
“package ‘tidyr’ was built under R version 4.5.2”
Warning message:
“package ‘ggplot2’ was built under R version 4.5.3”
Warning message:
“package ‘tibble’ was built under R version 4.5.2”
Warning message:
“package ‘readr’ was built under R version 4.5.2”
Warning message:
“package ‘forcats’ was built under R version 4.5.2”


## Metadata And Study Labels

This section reads metadata and creates the analysis labels. `study_id` and `chmi_qPCR` are combined into the six modeled crossed labels `EG_P`, `EG_NP`, `TU_P`, `TU_NP`, `GA_P`, and `GA_NP`. Check the tables here before interpreting models, because class imbalance and missing metadata define the limits of all downstream claims.


In [3]:
# Metadata helpers -------------------------------------------------------------
read_metadata <- function(path) {
  if (!file.exists(path)) {
    stop("Missing metadata file: ", path, call. = FALSE)
  }
  meta <- read.csv(path, row.names = 1, check.names = FALSE)
  if ("sample_id" %in% names(meta) && identical(rownames(meta), as.character(seq_len(nrow(meta))))) {
    rownames(meta) <- as.character(meta$sample_id)
  }
  if ("sample_id" %in% names(meta)) {
    meta$sample_id <- NULL
  }
  meta
}

add_analysis_labels <- function(meta) {
  meta <- as.data.frame(meta, check.names = FALSE)
  meta$protection <- dplyr::case_when(
    as.character(meta$chmi_qPCR) == "No" ~ "P",
    as.character(meta$chmi_qPCR) == "Yes" ~ "NP",
    TRUE ~ NA_character_
  )
  meta$protection <- factor(meta$protection, levels = c("P", "NP"))
  meta$study_id <- factor(meta$study_id, levels = names(STUDY_GROUP_MAP))
  meta$study_arm <- unname(STUDY_GROUP_MAP[as.character(meta$study_id)])
  meta$study_arm <- factor(meta$study_arm, levels = STUDY_ARM_LEVELS)
  crossed <- ifelse(
    !is.na(meta$study_arm) & !is.na(meta$protection),
    paste(as.character(meta$study_arm), as.character(meta$protection), sep = "_"),
    NA_character_
  )
  meta$study_protection_group <- factor(crossed, levels = STUDY_GROUP_LEVELS)
  meta$study_group <- meta$study_protection_group
  meta$timepoint <- factor(meta$timepoint, levels = c("t1", "t2", "t3"))
  if ("age_in_years" %in% names(meta)) {
    meta$age_in_years <- as.numeric(gsub(",", ".", meta$age_in_years))
  }
  if ("bmi" %in% names(meta)) {
    meta$bmi <- as.numeric(gsub(",", ".", meta$bmi))
  }
  meta
}

metadata_raw <- read_metadata(METADATA_PATH)
metadata_labeled <- add_analysis_labels(metadata_raw)

metadata_label_qc <- metadata_labeled |>
  tibble::rownames_to_column("sample_id") |>
  dplyr::summarise(
    n_metadata_rows = dplyr::n(),
    n_baseline_rows = sum(.data$timepoint == "t1", na.rm = TRUE),
    n_mapped_study_arm = sum(!is.na(.data$study_arm)),
    n_p_np_labeled = sum(!is.na(.data$protection)),
    n_crossed_group_labeled = sum(!is.na(.data$study_protection_group)),
    n_pfga1_rows_excluded_from_analysis = sum(as.character(.data$study_id) == "PfGA1", na.rm = TRUE)
  )

baseline_label_qc <- metadata_labeled |>
  tibble::rownames_to_column("sample_id") |>
  dplyr::filter(.data$timepoint == "t1", !is.na(.data$study_protection_group)) |>
  dplyr::count(.data$study_arm, .data$protection, name = "n") |>
  tidyr::complete(study_arm = factor(STUDY_ARM_LEVELS, levels = STUDY_ARM_LEVELS),
                  protection = factor(c("P", "NP"), levels = c("P", "NP")),
                  fill = list(n = 0L))

baseline_crossed_label_qc <- metadata_labeled |>
  tibble::rownames_to_column("sample_id") |>
  dplyr::filter(.data$timepoint == "t1", !is.na(.data$study_protection_group)) |>
  dplyr::count(.data$study_protection_group, name = "n") |>
  tidyr::complete(study_protection_group = factor(STUDY_GROUP_LEVELS, levels = STUDY_GROUP_LEVELS),
                  fill = list(n = 0L))

metadata_label_qc
baseline_label_qc
baseline_crossed_label_qc


n_metadata_rows,n_baseline_rows,n_mapped_study_arm,n_p_np_labeled,n_crossed_group_labeled,n_pfga1_rows_excluded_from_analysis
<int>,<int>,<int>,<int>,<int>,<int>
174,54,144,168,144,0


study_arm,protection,n
<fct>,<fct>,<int>
EG,P,6
EG,NP,4
GA,P,9
GA,NP,10
TU,P,12
TU,NP,5


study_protection_group,n
<fct>,<int>
EG_P,6
EG_NP,4
TU_P,12
TU_NP,5
GA_P,9
GA_NP,10


## All-View RaJIVE-Style Preprocessing

This section applies the same preprocessing logic used before the RaJIVE analysis: align the sample universe across blocks, exclude `PfGA1`, impute missing block values if needed, remove near-zero variance features, robust-scale features, and normalize each block by its Frobenius norm. The expected output is one scaled matrix per immune view plus QC tables that document missingness, feature filtering, and scaling.


In [4]:
# RaJIVE-style all-view preprocessing helpers ---------------------------------
`%||%` <- function(x, y) if (is.null(x)) y else x

safe_natural_sort <- function(x) {
  if (requireNamespace("naturalsort", quietly = TRUE)) {
    naturalsort::naturalsort(x)
  } else {
    sort(x)
  }
}

check_input_files <- function(paths, metadata_path) {
  all_paths <- c(unname(paths), metadata_path)
  tibble::tibble(
    name = c(names(paths), "metadata"),
    path = all_paths,
    exists = file.exists(all_paths)
  )
}

read_original_blocks <- function(paths) {
  blocks <- lapply(paths, function(path) {
    if (!file.exists(path)) {
      stop("Missing feature matrix: ", path, call. = FALSE)
    }
    read.csv(path, row.names = 1, check.names = FALSE)
  })
  names(blocks) <- names(paths)
  blocks
}

make_analysis_samples <- function(blocks, metadata) {
  all_samples <- safe_natural_sort(Reduce(union, lapply(blocks, rownames)))
  pfga1 <- rownames(metadata)[!is.na(metadata$study_id) & metadata$study_id == "PfGA1"]
  setdiff(all_samples, pfga1)
}

summarize_original_qc <- function(blocks, metadata, analysis_samples) {
  sample_qc <- purrr::imap_dfr(blocks, function(x, nm) {
    tibble::tibble(
      view = nm,
      raw_samples = nrow(x),
      raw_features = ncol(x),
      samples_in_analysis = sum(analysis_samples %in% rownames(x)),
      samples_missing_from_view = sum(!analysis_samples %in% rownames(x)),
      raw_na_values = sum(is.na(x)),
      raw_na_samples = sum(rowSums(is.na(x)) > 0),
      raw_na_features = sum(colSums(is.na(x)) > 0)
    )
  })

  metadata_qc <- tibble::tibble(
    n_analysis_samples = length(analysis_samples),
    n_metadata_rows = nrow(metadata),
    n_samples_missing_metadata = sum(!analysis_samples %in% rownames(metadata)),
    n_pfga1_rows_excluded = sum(!is.na(metadata$study_id) & metadata$study_id == "PfGA1")
  )

  list(sample_qc = sample_qc, metadata_qc = metadata_qc)
}

align_and_impute_blocks <- function(blocks, analysis_samples, seed = 1L) {
  if (!requireNamespace("missForest", quietly = TRUE)) {
    stop("Package 'missForest' is required when running RaJIVE-style imputation.", call. = FALSE)
  }
  set.seed(seed)
  imputed <- purrr::imap(blocks, function(x, nm) {
    x <- as.data.frame(x, check.names = FALSE)
    missing_samples <- setdiff(analysis_samples, rownames(x))
    if (length(missing_samples) > 0L) {
      add <- as.data.frame(matrix(NA_real_, nrow = length(missing_samples), ncol = ncol(x),
                                  dimnames = list(missing_samples, colnames(x))))
      x <- rbind(x, add)
    }
    x <- x[analysis_samples, , drop = FALSE]
    colnames(x) <- make.unique(colnames(x), sep = "__dup")
    if (anyNA(x)) {
      x <- missForest::missForest(x, verbose = TRUE)$ximp
    }
    out <- as.matrix(x)
    storage.mode(out) <- "double"
    rownames(out) <- analysis_samples
    out
  })
  names(imputed) <- names(blocks)
  imputed
}

filter_near_zero_variance_blocks <- function(blocks, freqCut = 95 / 5, uniqueCut = 10,
                                             max_removed_examples = 8L) {
  if (!requireNamespace("mixOmics", quietly = TRUE)) {
    stop("Package 'mixOmics' is required for near-zero variance filtering.", call. = FALSE)
  }
  summaries <- list()
  filtered <- purrr::imap(blocks, function(x, nm) {
    x <- as.matrix(x)
    storage.mode(x) <- "double"
    nzv <- mixOmics::nearZeroVar(x, freqCut = freqCut, uniqueCut = uniqueCut)
    remove_idx <- if (is.null(nzv$Position)) integer(0) else as.integer(nzv$Position)
    remove_idx <- remove_idx[!is.na(remove_idx) & remove_idx >= 1L & remove_idx <= ncol(x)]
    keep_idx <- setdiff(seq_len(ncol(x)), remove_idx)
    if (length(keep_idx) == 0L) {
      stop("View '", nm, "' has no features left after near-zero variance filtering.", call. = FALSE)
    }
    removed_features <- colnames(x)[remove_idx]
    summaries[[nm]] <<- tibble::tibble(
      view = nm,
      n_features_before_nzv = ncol(x),
      n_near_zero_variance_features = length(remove_idx),
      n_features_after_nzv = length(keep_idx),
      removed_feature_examples = paste(utils::head(removed_features, max_removed_examples), collapse = ", ")
    )
    x[, keep_idx, drop = FALSE]
  })
  names(filtered) <- names(blocks)
  list(blocks = filtered, summary = dplyr::bind_rows(summaries))
}

scale_blocks_robust_frob <- function(blocks, center_only_blocks = "cytof_celltype") {
  summaries <- list()
  scaled <- purrr::imap(blocks, function(x, nm) {
    x <- as.matrix(x)
    storage.mode(x) <- "double"
    med <- apply(x, 2L, stats::median, na.rm = TRUE)
    madv <- apply(x, 2L, stats::mad, na.rm = TRUE)
    zero_mad <- !is.finite(madv) | madv <= .Machine$double.eps

    centered <- sweep(x, 2L, med, "-")
    if (nm %in% center_only_blocks) {
      z <- centered
      scaling_mode <- "center_only"
    } else {
      z <- centered
      if (any(!zero_mad)) {
        z[, !zero_mad] <- sweep(z[, !zero_mad, drop = FALSE], 2L, madv[!zero_mad], "/")
      }
      if (any(zero_mad)) {
        z[, zero_mad] <- 0
      }
      scaling_mode <- "median_mad"
    }

    frob_before <- norm(z, type = "F")
    if (!is.finite(frob_before) || frob_before <= .Machine$double.eps) {
      stop("View '", nm, "' has zero Frobenius norm after robust scaling.", call. = FALSE)
    }
    z <- z / frob_before
    attr(z, "frob.scale.val") <- frob_before
    summaries[[nm]] <<- tibble::tibble(
      view = nm,
      scaling_mode = scaling_mode,
      n_samples = nrow(x),
      n_features = ncol(x),
      zero_mad_features = sum(zero_mad),
      frob_before = frob_before,
      frob_after = norm(z, type = "F")
    )
    z
  })
  names(scaled) <- names(blocks)
  list(blocks = scaled, summary = dplyr::bind_rows(summaries))
}

feature_name_audit <- function(blocks, max_examples = 8L) {
  purrr::imap_dfr(blocks, function(x, nm) {
    tibble::tibble(
      view = nm,
      n_features = ncol(x),
      n_duplicate_feature_names = sum(duplicated(colnames(x))),
      first_features = paste(utils::head(colnames(x), max_examples), collapse = ", ")
    )
  })
}

subset_baseline_all_views <- function(blocks, metadata) {
  meta <- add_analysis_labels(metadata)
  eligible <- rownames(meta)[
    !is.na(meta$timepoint) & meta$timepoint == "t1" &
      !is.na(meta$study_group)
  ]
  common_ids <- Reduce(intersect, c(list(eligible), lapply(blocks, rownames)))
  common_ids <- eligible[eligible %in% common_ids]
  x_list <- lapply(blocks, function(x) x[common_ids, , drop = FALSE])
  meta_out <- droplevels(meta[common_ids, , drop = FALSE])
  y_out <- stats::setNames(meta_out$study_group, common_ids)
  if (any(vapply(x_list, function(x) anyNA(x), logical(1L)))) {
    stop("Baseline all-view blocks contain missing values after preprocessing.", call. = FALSE)
  }
  if (any(!vapply(x_list, function(x) identical(rownames(x), names(y_out)), logical(1L)))) {
    stop("Baseline all-view sample alignment failed.", call. = FALSE)
  }
  list(x_list = x_list, y = y_out, metadata = meta_out)
}

prefix_feature_names <- function(x, view) {
  x <- as.matrix(x)
  colnames(x) <- paste(view, colnames(x), sep = "__")
  x
}

split_prefixed_feature <- function(feature) {
  parts <- strsplit(feature, "__", fixed = TRUE)
  tibble::tibble(
    feature = feature,
    view = vapply(parts, `[`, character(1L), 1L),
    feature_name = vapply(parts, function(x) paste(x[-1L], collapse = "__"), character(1L))
  )
}


In [5]:
preprocess_cache_candidates <- c(
  notebook_branch = cached_rds_path("preprocess", "rajive_style_preprocessed_all_views"),
  repo_legacy = file.path(CACHE_DIR, "rajive_style_preprocessed_all_views.rds"),
  slurm_baseline = cached_rds_path("preprocess", "baseline_preprocessed")
)
preprocess_cache_path <- first_existing_path(preprocess_cache_candidates)
baseline_preprocessed_cache <- NULL

if (isTRUE(LOAD_CACHED_RESULTS) && !isTRUE(FORCE_RECOMPUTE) && !is.na(preprocess_cache_path)) {
  preprocess_cache <- readRDS(preprocess_cache_path)
  if (!is.null(preprocess_cache$scaled_blocks)) {
    for (nm in names(preprocess_cache)) {
      assign(nm, preprocess_cache[[nm]], envir = .GlobalEnv)
    }
    message("Loaded RaJIVE-style preprocessing cache: ", preprocess_cache_path)
  } else if (all(c("x_list", "y", "metadata") %in% names(preprocess_cache))) {
    baseline_preprocessed_cache <- preprocess_cache
    analysis_samples <- preprocess_cache$analysis_samples %||% names(preprocess_cache$y)
    input_file_qc <- load_cached_table("preprocess", "input_file_qc")
    original_qc <- list(
      metadata_qc = load_cached_table("preprocess", "original_metadata_qc"),
      sample_qc = load_cached_table("preprocess", "original_sample_qc")
    )
    raw_feature_name_qc <- load_cached_table("preprocess", "raw_feature_name_qc")
    near_zero_variance_qc <- preprocess_cache$nzv_summary %||% load_cached_table("preprocess", "near_zero_variance_qc")
    scaling_qc <- preprocess_cache$scaling_qc %||% load_cached_table("preprocess", "scaling_qc")
    message("Loaded baseline preprocessing cache: ", preprocess_cache_path)
  } else {
    stop("Unsupported preprocessing cache format: ", preprocess_cache_path, call. = FALSE)
  }
} else {
  input_file_qc <- check_input_files(ALL_VIEW_PATHS, METADATA_PATH)
  stopifnot(all(input_file_qc$exists))

  raw_blocks <- read_original_blocks(ALL_VIEW_PATHS)
  analysis_samples <- make_analysis_samples(raw_blocks, metadata_raw)
  original_qc <- summarize_original_qc(raw_blocks, metadata_raw, analysis_samples)
  raw_feature_name_qc <- feature_name_audit(raw_blocks)

  set.seed(IMPUTATION_SEED)
  aligned_blocks <- align_and_impute_blocks(raw_blocks, analysis_samples, seed = IMPUTATION_SEED)
  nzv_result <- filter_near_zero_variance_blocks(aligned_blocks)
  filtered_blocks <- nzv_result$blocks
  near_zero_variance_qc <- nzv_result$summary

  scaled_result <- scale_blocks_robust_frob(filtered_blocks, center_only_blocks = "cytof_celltype")
  scaled_blocks <- scaled_result$blocks
  scaling_qc <- scaled_result$summary

  preprocess_cache <- list(
    input_file_qc = input_file_qc,
    analysis_samples = analysis_samples,
    original_qc = original_qc,
    raw_feature_name_qc = raw_feature_name_qc,
    near_zero_variance_qc = near_zero_variance_qc,
    scaling_qc = scaling_qc,
    scaled_blocks = scaled_blocks
  )
  cache_object(preprocess_cache, "rajive_style_preprocessed_all_views", branch = "preprocess")
}

if (exists("input_file_qc") && nrow(input_file_qc) > 0L) export_table(input_file_qc, "input_file_qc", branch = "preprocess")
if (exists("original_qc") && !is.null(original_qc$metadata_qc) && nrow(original_qc$metadata_qc) > 0L) export_table(original_qc$metadata_qc, "original_metadata_qc", branch = "preprocess")
if (exists("original_qc") && !is.null(original_qc$sample_qc) && nrow(original_qc$sample_qc) > 0L) export_table(original_qc$sample_qc, "original_sample_qc", branch = "preprocess")
if (exists("raw_feature_name_qc") && nrow(raw_feature_name_qc) > 0L) export_table(raw_feature_name_qc, "raw_feature_name_qc", branch = "preprocess")
if (exists("near_zero_variance_qc") && nrow(near_zero_variance_qc) > 0L) export_table(near_zero_variance_qc, "near_zero_variance_qc", branch = "preprocess")
if (exists("scaling_qc") && nrow(scaling_qc) > 0L) export_table(scaling_qc, "scaling_qc", branch = "preprocess")

if (exists("input_file_qc") && "exists" %in% names(input_file_qc)) stopifnot(all(input_file_qc$exists))
if (exists("scaled_blocks")) {
  stopifnot(all(vapply(scaled_blocks, function(x) identical(rownames(x), analysis_samples), logical(1L))))
  stopifnot(all(abs(vapply(scaled_blocks, norm, numeric(1L), type = "F") - 1) < 1e-8))
} else if (!is.null(baseline_preprocessed_cache)) {
  stopifnot(all(vapply(baseline_preprocessed_cache$x_list, function(x) !anyNA(x), logical(1L))))
}

input_file_qc
original_qc$metadata_qc
original_qc$sample_qc
near_zero_variance_qc
scaling_qc


Loaded baseline preprocessing cache: /exports/para-lipg-hpc/mdmanurung/stablr/scratch/cache/stablr_baseline_study_protection_test/preprocess/baseline_preprocessed.rds



<0 x 0 matrix>

<0 x 0 matrix>

<0 x 0 matrix>

view,n_features_before_nzv,n_near_zero_variance_features,n_features_after_nzv
<chr>,<int>,<int>,<int>
cytof_celltype,125,0,125
exvivo_celltype,40,0,40
exvivo_enzyme,98,0,98
6h_cyto_LPS,130,0,130
6h_cyto_ssRNA40,130,0,130
24h_cyto_iRBC,166,9,157
24h_cyto_SEB,166,1,165
24h_enzyme_iRBC,94,0,94
24h_enzyme_SEB,99,0,99


view,scaling_mode,n_samples,n_features,zero_mad_features,frob_before,frob_after
<chr>,<chr>,<int>,<int>,<int>,<dbl>,<dbl>
cytof_celltype,center_only,120,125,0,104.23657,1
exvivo_celltype,median_mad,120,40,0,83.67294,1
exvivo_enzyme,median_mad,120,98,0,150.22776,1
6h_cyto_LPS,median_mad,120,130,0,199.77942,1
6h_cyto_ssRNA40,median_mad,120,130,1,254.41065,1
24h_cyto_iRBC,median_mad,120,157,5,280.30582,1
24h_cyto_SEB,median_mad,120,165,0,254.64499,1
24h_enzyme_iRBC,median_mad,120,94,0,115.70210,1
24h_enzyme_SEB,median_mad,120,99,0,134.13575,1


## Baseline Cohort QC

This section restricts the preprocessed matrices to baseline samples and checks that all 11 views are aligned. The key outputs are class counts, P/NP counts within study/protection group, and retained feature counts per view. Interpret downstream STABL results in light of this small `n = 38` cohort and the three-class imbalance.


In [6]:
if (!is.null(baseline_preprocessed_cache)) {
  x_all_list <- baseline_preprocessed_cache$x_list
  y_all <- baseline_preprocessed_cache$y
  metadata_baseline <- baseline_preprocessed_cache$metadata
  baseline_all_views <- list(x_list = x_all_list, y = y_all, metadata = metadata_baseline)
} else {
  baseline_all_views <- subset_baseline_all_views(scaled_blocks, metadata_raw)
  x_all_list <- baseline_all_views$x_list
  y_all <- baseline_all_views$y
  metadata_baseline <- baseline_all_views$metadata
}

bootstrap_strata_all <- data.frame(study_protection_group = y_all, row.names = names(y_all))

baseline_class_qc <- tibble::tibble(study_protection_group = y_all) |>
  dplyr::count(.data$study_protection_group, name = "n") |>
  tidyr::complete(study_protection_group = factor(STUDY_GROUP_LEVELS, levels = STUDY_GROUP_LEVELS), fill = list(n = 0L))

baseline_protection_qc <- as.data.frame(metadata_baseline, check.names = FALSE) |>
  dplyr::count(.data$study_arm, .data$protection, name = "n")

baseline_view_qc <- purrr::imap_dfr(x_all_list, function(x, view) {
  tibble::tibble(
    view = view,
    n_samples = nrow(x),
    n_features = ncol(x),
    n_missing = sum(is.na(x)),
    frobenius_norm = norm(x, type = "F")
  )
})

if (length(y_all) != EXPECTED_BASELINE_N) {
  warning("Expected ", EXPECTED_BASELINE_N, " baseline samples, found ", length(y_all), call. = FALSE)
}
observed_counts <- table(y_all)
if (!all(names(EXPECTED_BASELINE_COUNTS) %in% names(observed_counts)) ||
    any(as.integer(observed_counts[names(EXPECTED_BASELINE_COUNTS)]) != as.integer(EXPECTED_BASELINE_COUNTS))) {
  warning("Baseline crossed-label balance differs from expectation: ",
          paste(names(observed_counts), observed_counts, sep = "=", collapse = ", "), call. = FALSE)
}

export_table(baseline_class_qc, "baseline_class_qc", branch = "preprocess")
export_table(baseline_protection_qc, "baseline_arm_protection_qc", branch = "preprocess")
export_table(baseline_view_qc, "baseline_view_qc", branch = "preprocess")
cache_object(list(x_list = x_all_list, y = y_all, metadata = metadata_baseline), "baseline_complete_case_views", branch = "preprocess")

baseline_class_qc
baseline_protection_qc
baseline_view_qc


study_protection_group,n
<fct>,<int>
EG_P,6
EG_NP,4
TU_P,9
TU_NP,3
GA_P,8
GA_NP,8


study_arm,protection,n
<fct>,<fct>,<int>
EG,P,6
EG,NP,4
GA,P,8
GA,NP,8
TU,P,9
TU,NP,3


view,n_samples,n_features,n_missing,frobenius_norm
<chr>,<int>,<int>,<int>,<dbl>
cytof_celltype,38,125,0,0.5692460
exvivo_celltype,38,40,0,0.5401055
exvivo_enzyme,38,98,0,0.5694408
6h_cyto_LPS,38,130,0,0.5570706
6h_cyto_ssRNA40,38,130,0,0.5843107
24h_cyto_iRBC,38,157,0,0.4690036
24h_cyto_SEB,38,165,0,0.5381928
24h_enzyme_iRBC,38,94,0,0.5260319
24h_enzyme_SEB,38,99,0,0.5374754


## STABL And Interpretation Helpers

This section defines the reusable modeling and plotting functions. The helpers return tidy stability tables, class-specific elastic-net coefficients, standardized group summaries, and fallback feature sets when no feature crosses the STABL threshold. These objects are the basis for interpreting immune signatures rather than only listing selected features.


In [7]:
# STABL and interpretation helpers -------------------------------------------
fit_stabl_elastic_net <- function(x, y, seed, view_name = "view", bootstrap_strata = NULL, family = STABL_FAMILY) {
  lambda_grid <- auto_lambda_grid(
    x, y,
    family = family,
    n_lambda = N_LAMBDA,
    l1_ratio = ELASTIC_NET_L1_RATIO
  )

  fit <- stabl_fit(
    x = x,
    y = y,
    lambda_grid = lambda_grid,
    base_learner = STABL_BASE_LEARNER,
    family = family,
    n_bootstraps = N_BOOTSTRAPS,
    artificial_type = ARTIFICIAL_TYPE,
    artificial_proportion = ARTIFICIAL_PROPORTION,
    sample_fraction = SAMPLE_FRACTION,
    replace = FALSE,
    bootstrap_strata = bootstrap_strata,
    random_state = seed,
    workers = N_WORKERS,
    fdr_threshold_range = FDR_THRESHOLD_RANGE
  )

  list(view = view_name, lambda_grid = lambda_grid, fit = fit)
}

importance_table <- function(fit) {
  scores <- get_importances(fit)
  selected <- get_feature_names_out(fit)
  tibble::tibble(
    feature = names(scores),
    stability_score = as.numeric(scores),
    selected = names(scores) %in% selected
  ) |>
    dplyr::arrange(dplyr::desc(.data$selected), dplyr::desc(.data$stability_score), .data$feature)
}

choose_features_for_group_plot <- function(fit, top_n = TOP_FALLBACK_FEATURES) {
  selected <- get_feature_names_out(fit)
  scores <- sort(get_importances(fit), decreasing = TRUE)
  if (length(selected) > 0L) {
    return(list(features = selected, fallback = FALSE))
  }
  fallback_features <- names(scores)[seq_len(min(top_n, length(scores)))]
  list(features = fallback_features, fallback = TRUE)
}

multinomial_elastic_net_beta_table <- function(x, y, lambda_grid, alpha = ELASTIC_NET_L1_RATIO) {
  if (!requireNamespace("glmnet", quietly = TRUE)) {
    stop("Package 'glmnet' is required for coefficient extraction.", call. = FALSE)
  }
  lambdas <- sort(unique(lambda_grid$lambda), decreasing = TRUE)
  beta_fit <- glmnet::glmnet(
    x = x,
    y = y,
    family = "multinomial",
    alpha = alpha,
    lambda = lambdas
  )
  groups <- levels(droplevels(y))

  purrr::map_dfr(groups, function(group) {
    beta_by_lambda <- purrr::map_dfr(lambdas, function(lambda_value) {
      coef_list <- glmnet::coef.glmnet(beta_fit, s = lambda_value)
      if (!group %in% names(coef_list)) {
        stop("Missing multinomial coefficient block for group: ", group, call. = FALSE)
      }
      coef_mat <- coef_list[[group]]
      tibble::tibble(
        study_group = group,
        feature = rownames(coef_mat)[-1L],
        lambda = lambda_value,
        beta = as.numeric(coef_mat[-1L, 1L])
      )
    })

    beta_by_lambda |>
      dplyr::group_by(.data$study_group, .data$feature) |>
      dplyr::slice_max(order_by = abs(.data$beta), n = 1L, with_ties = FALSE) |>
      dplyr::ungroup() |>
      dplyr::transmute(
        study_group = .data$study_group,
        feature = .data$feature,
        beta = .data$beta,
        abs_beta = abs(.data$beta),
        lambda_at_max_abs_beta = .data$lambda
      )
  })
}

feature_group_summary <- function(x, y, features = colnames(x)) {
  features <- intersect(features, colnames(x))
  values <- as.data.frame(x[, features, drop = FALSE], check.names = FALSE) |>
    tibble::rownames_to_column("sample_id") |>
    tidyr::pivot_longer(cols = -dplyr::all_of("sample_id"), names_to = "feature", values_to = "value") |>
    dplyr::mutate(study_group = as.character(y[.data$sample_id]))

  values |>
    dplyr::group_by(.data$feature, .data$study_group) |>
    dplyr::summarise(
      group_mean = mean(.data$value, na.rm = TRUE),
      group_median = stats::median(.data$value, na.rm = TRUE),
      group_sd = stats::sd(.data$value, na.rm = TRUE),
      n = dplyr::n(),
      .groups = "drop"
    ) |>
    dplyr::group_by(.data$feature) |>
    dplyr::mutate(
      rest_mean = (sum(.data$group_mean * .data$n) - .data$group_mean * .data$n) / (sum(.data$n) - .data$n),
      one_vs_rest_mean_diff = .data$group_mean - .data$rest_mean
    ) |>
    dplyr::ungroup()
}

immune_signature_table <- function(fit, x, y, lambda_grid, view = NULL, top_n = NULL) {
  stability <- importance_table(fit)
  beta_tbl <- multinomial_elastic_net_beta_table(x, y, lambda_grid)
  group_summary <- feature_group_summary(x, y)

  out <- beta_tbl |>
    dplyr::left_join(stability, by = "feature") |>
    dplyr::left_join(group_summary, by = c("feature", "study_group")) |>
    dplyr::mutate(
      view = view %||% "model",
      stability_score = tidyr::replace_na(.data$stability_score, 0),
      selected = tidyr::replace_na(.data$selected, FALSE),
      rank_score = .data$stability_score * .data$abs_beta
    ) |>
    dplyr::select(.data$view, .data$study_group, .data$feature, .data$selected,
                  .data$stability_score, .data$beta, .data$abs_beta,
                  .data$rank_score, .data$group_mean, .data$rest_mean,
                  .data$one_vs_rest_mean_diff, .data$lambda_at_max_abs_beta) |>
    dplyr::arrange(.data$study_group, dplyr::desc(.data$rank_score), dplyr::desc(.data$stability_score))

  if (!is.null(top_n)) {
    out <- out |>
      dplyr::group_by(.data$study_group) |>
      dplyr::slice_head(n = top_n) |>
      dplyr::ungroup()
  }
  out
}

plot_top_predictors_by_study_group <- function(top_tbl, title = "Top predictors by study group") {
  plot_tbl <- top_tbl |>
    dplyr::group_by(.data$study_group) |>
    dplyr::arrange(.data$rank_score, .by_group = TRUE) |>
    dplyr::mutate(feature_panel = paste(.data$study_group, .data$feature, sep = " || ")) |>
    dplyr::ungroup()
  plot_tbl$feature_panel <- factor(plot_tbl$feature_panel, levels = unique(plot_tbl$feature_panel))

  ggplot2::ggplot(plot_tbl, ggplot2::aes(x = .data$stability_score, y = .data$feature_panel)) +
    ggplot2::geom_segment(ggplot2::aes(x = 0, xend = .data$stability_score, yend = .data$feature_panel), color = "grey80") +
    ggplot2::geom_point(ggplot2::aes(size = .data$abs_beta, color = .data$beta), alpha = 0.9) +
    ggplot2::facet_wrap(~ study_group, scales = "free_y") +
    ggplot2::scale_x_continuous(limits = c(0, 1), expand = ggplot2::expansion(mult = c(0, 0.04))) +
    ggplot2::scale_y_discrete(labels = function(x) sub("^.* \\|\\| ", "", x)) +
    ggplot2::scale_color_gradient2(low = "#B2182B", mid = "grey75", high = "#2166AC", midpoint = 0) +
    ggplot2::labs(
      title = title,
      subtitle = "Ranked by |class beta| x STABL stability; point size is |beta| and color is signed beta.",
      x = "STABL stability score",
      y = NULL,
      size = "|beta|",
      color = "beta"
    ) +
    ggplot2::theme_bw(base_size = 10) +
    ggplot2::theme(
      legend.position = "top",
      panel.grid.minor = ggplot2::element_blank(),
      strip.text = ggplot2::element_text(face = "bold")
    )
}

plot_features_by_study_class <- function(features, x, metadata, fallback = FALSE, title = NULL) {
  features <- intersect(features, colnames(x))
  if (length(features) == 0L) {
    stop("No requested features are present in the predictor matrix.", call. = FALSE)
  }

  feature_values <- as.data.frame(x[, features, drop = FALSE], check.names = FALSE) |>
    tibble::rownames_to_column("sample_id") |>
    tidyr::pivot_longer(cols = -dplyr::all_of("sample_id"), names_to = "feature", values_to = "value")

  meta_plot <- as.data.frame(metadata, check.names = FALSE) |>
    tibble::rownames_to_column("sample_id") |>
    dplyr::select(dplyr::all_of(c("sample_id", "study_group", "protection")))

  plot_df <- feature_values |>
    dplyr::left_join(meta_plot, by = "sample_id") |>
    dplyr::filter(!is.na(.data$study_group))

  if (is.null(title)) {
    title <- if (fallback) "Top stability features by baseline study group" else "Selected STABL features by baseline study group"
  }
  subtitle <- if (fallback) {
    "Exploratory fallback: no features crossed the STABL threshold; plotting top-ranked features."
  } else {
    "P/NP is shown only as descriptive QC; the model target is study group."
  }

  p <- ggplot2::ggplot(plot_df, ggplot2::aes(x = .data$study_group, y = .data$value, fill = .data$study_group)) +
    ggplot2::geom_boxplot(width = 0.6, alpha = 0.8, outlier.shape = NA) +
    ggplot2::geom_jitter(ggplot2::aes(shape = .data$protection), width = 0.12, size = 1.2, alpha = 0.75) +
    ggplot2::facet_wrap(~ feature, scales = "free_y") +
    ggplot2::labs(title = title, subtitle = subtitle, x = NULL, y = "Preprocessed feature value", fill = "Study", shape = "P/NP") +
    ggplot2::scale_fill_manual(values = STUDY_GROUP_COLORS, drop = FALSE) +
    ggplot2::theme_bw(base_size = 10) +
    ggplot2::theme(
      legend.position = "top",
      panel.grid.minor = ggplot2::element_blank(),
      strip.text.x = ggplot2::element_text(size = 7),
      axis.text.x = ggplot2::element_text(angle = 0, hjust = 0.5)
    )

  if (requireNamespace("ggpubr", quietly = TRUE)) {
    p <- p + ggpubr::stat_pwc(method = "t_test", p.adjust.method = "none", hide.ns = TRUE)
  }
  p
}

plot_group_mean_heatmap <- function(signature_tbl, title = "Group-level immune signature summary") {
  heat_tbl <- signature_tbl |>
    dplyr::distinct(.data$study_group, .data$feature, .data$group_mean, .data$selected) |>
    dplyr::mutate(feature = forcats::fct_reorder(.data$feature, .data$group_mean, .fun = max))

  ggplot2::ggplot(heat_tbl, ggplot2::aes(x = .data$study_group, y = .data$feature, fill = .data$group_mean)) +
    ggplot2::geom_tile(color = "white", linewidth = 0.2) +
    ggplot2::scale_fill_gradient2(low = "#B2182B", mid = "white", high = "#2166AC", midpoint = 0) +
    ggplot2::labs(
      title = title,
      subtitle = "Values are group means after RaJIVE-style preprocessing; compare direction across groups, not absolute biology.",
      x = NULL,
      y = NULL,
      fill = "Mean"
    ) +
    ggplot2::theme_bw(base_size = 10) +
    ggplot2::theme(panel.grid = ggplot2::element_blank(), legend.position = "top")
}

plot_view_contribution <- function(signature_tbl, title = "Cross-view contribution to study/protection signatures") {
  signature_tbl |>
    dplyr::mutate(view = sub("__.*$", "", .data$feature)) |>
    dplyr::group_by(.data$study_group, .data$view) |>
    dplyr::summarise(
      n_features = dplyr::n_distinct(.data$feature),
      max_stability = max(.data$stability_score, na.rm = TRUE),
      total_rank_score = sum(.data$rank_score, na.rm = TRUE),
      .groups = "drop"
    ) |>
    ggplot2::ggplot(ggplot2::aes(x = forcats::fct_reorder(.data$view, .data$total_rank_score, .fun = sum),
                                 y = .data$total_rank_score, fill = .data$study_group)) +
    ggplot2::geom_col(position = "dodge") +
    ggplot2::coord_flip() +
    ggplot2::scale_fill_manual(values = STUDY_GROUP_COLORS, drop = FALSE) +
    ggplot2::labs(
      title = title,
      subtitle = "Higher bars indicate views contributing more high-stability, high-beta predictors.",
      x = NULL,
      y = "Sum of stability x |beta|",
      fill = "Study"
    ) +
    ggplot2::theme_bw(base_size = 10) +
    ggplot2::theme(legend.position = "top", panel.grid.minor = ggplot2::element_blank())
}

classification_metric_table <- function(truth, predicted, levels = STUDY_GROUP_LEVELS) {
  truth <- factor(truth, levels = levels)
  predicted <- factor(predicted, levels = levels)
  confusion <- table(truth = truth, predicted = predicted)
  recall <- diag(confusion) / pmax(rowSums(confusion), 1L)
  precision <- diag(confusion) / pmax(colSums(confusion), 1L)
  f1 <- ifelse(precision + recall > 0, 2 * precision * recall / (precision + recall), 0)
  tibble::tibble(
    metric = c("accuracy", "balanced_error_rate", "macro_f1"),
    value = c(
      mean(truth == predicted, na.rm = TRUE),
      mean(1 - recall, na.rm = TRUE),
      mean(f1)
    )
  )
}

per_class_metric_table <- function(truth, predicted, levels = STUDY_GROUP_LEVELS) {
  truth <- factor(truth, levels = levels)
  predicted <- factor(predicted, levels = levels)
  confusion <- table(truth = truth, predicted = predicted)
  recall <- diag(confusion) / pmax(rowSums(confusion), 1L)
  precision <- diag(confusion) / pmax(colSums(confusion), 1L)
  f1 <- ifelse(precision + recall > 0, 2 * precision * recall / (precision + recall), 0)
  tibble::tibble(
    class = levels,
    support = as.integer(rowSums(confusion)),
    predicted = as.integer(colSums(confusion)),
    recall = as.numeric(recall),
    precision = as.numeric(precision),
    f1 = as.numeric(f1)
  )
}

confusion_table <- function(truth, predicted, levels = STUDY_GROUP_LEVELS) {
  as.data.frame(table(
    truth = factor(truth, levels = levels),
    predicted = factor(predicted, levels = levels)
  ))
}

choose_features_for_heatmap <- function(fit, top_n = TOP_HEATMAP_FEATURES) {
  scores <- sort(get_importances(fit), decreasing = TRUE)
  selected <- get_feature_names_out(fit)
  fallback <- length(selected) == 0L
  features <- if (fallback) names(scores)[seq_len(min(top_n, length(scores)))] else selected
  tibble::tibble(
    feature = features,
    stability_score = as.numeric(scores[features]),
    selected = features %in% selected,
    exploratory_fallback = fallback
  )
}

make_feature_heatmap_matrix <- function(x, features) {
  features <- intersect(features, colnames(x))
  mat <- t(x[, features, drop = FALSE])
  mat <- t(scale(t(mat)))
  mat[!is.finite(mat)] <- 0
  mat
}

export_selected_feature_heatmap <- function(mat, sample_metadata, feature_metadata, filename,
                                            branch = infer_artifact_branch(filename),
                                            row_split = NULL, width = 12, height = 9,
                                            cluster_rows = TRUE, cluster_columns = TRUE) {
  sample_metadata <- as.data.frame(sample_metadata, check.names = FALSE)
  sample_metadata <- sample_metadata[colnames(mat), , drop = FALSE]
  feature_metadata <- as.data.frame(feature_metadata, check.names = FALSE)
  feature_metadata <- feature_metadata[match(rownames(mat), feature_metadata$feature), , drop = FALSE]
  fig_dir <- branch_export_dir(branch, "figures")
  png_path <- file.path(fig_dir, paste0(artifact_name(filename), ".png"))
  pdf_path <- file.path(fig_dir, paste0(artifact_name(filename), ".pdf"))

  if (requireNamespace("ComplexHeatmap", quietly = TRUE) && requireNamespace("circlize", quietly = TRUE)) {
    sex_annotation <- if ("sex" %in% names(sample_metadata)) sample_metadata$sex else rep(NA_character_, nrow(sample_metadata))
    top_ha <- ComplexHeatmap::HeatmapAnnotation(
      study_group = sample_metadata$study_group,
      study_id = sample_metadata$study_id,
      protection = sample_metadata$protection,
      sex = sex_annotation,
      col = list(study_group = STUDY_GROUP_COLORS)
    )
    left_ha <- ComplexHeatmap::rowAnnotation(
      view = feature_metadata$view,
      selected = feature_metadata$selected
    )
    ht <- ComplexHeatmap::Heatmap(
      mat,
      name = "z",
      top_annotation = top_ha,
      left_annotation = left_ha,
      cluster_rows = cluster_rows,
      cluster_columns = cluster_columns,
      row_split = row_split,
      show_column_names = FALSE,
      column_title = "Baseline samples",
      row_title = "Selected or fallback features"
    )
    grDevices::png(png_path, width = width * 200, height = height * 200, res = 200)
    ComplexHeatmap::draw(ht)
    grDevices::dev.off()
    grDevices::pdf(pdf_path, width = width, height = height)
    ComplexHeatmap::draw(ht)
    grDevices::dev.off()
    record_artifact(branch, "png", artifact_name(filename), png_path)
    record_artifact(branch, "pdf", artifact_name(filename), pdf_path)
  } else if (requireNamespace("pheatmap", quietly = TRUE)) {
    annotation_col <- sample_metadata[, intersect(c("study_group", "study_id", "protection", "sex"), names(sample_metadata)), drop = FALSE]
    annotation_row <- feature_metadata[, intersect(c("view", "selected"), names(feature_metadata)), drop = FALSE]
    rownames(annotation_col) <- colnames(mat)
    rownames(annotation_row) <- rownames(mat)
    pheatmap::pheatmap(mat, annotation_col = annotation_col, annotation_row = annotation_row,
                       cluster_rows = cluster_rows, cluster_cols = cluster_columns,
                       show_colnames = FALSE, filename = png_path, width = width, height = height)
    pheatmap::pheatmap(mat, annotation_col = annotation_col, annotation_row = annotation_row,
                       cluster_rows = cluster_rows, cluster_cols = cluster_columns,
                       show_colnames = FALSE, filename = pdf_path, width = width, height = height)
    record_artifact(branch, "png", artifact_name(filename), png_path)
    record_artifact(branch, "pdf", artifact_name(filename), pdf_path)
  } else {
    heat_df <- as.data.frame(mat, check.names = FALSE) |>
      tibble::rownames_to_column("feature") |>
      tidyr::pivot_longer(cols = -dplyr::all_of("feature"), names_to = "sample_id", values_to = "z")
    p <- ggplot2::ggplot(heat_df, ggplot2::aes(.data$sample_id, .data$feature, fill = .data$z)) +
      ggplot2::geom_tile() +
      ggplot2::scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#B2182B") +
      ggplot2::labs(x = "Sample", y = NULL, fill = "z") +
      ggplot2::theme_bw(base_size = 9) +
      ggplot2::theme(axis.text.x = ggplot2::element_blank(), axis.ticks.x = ggplot2::element_blank())
    export_plot(p, filename, branch = branch, width = width, height = height)
  }
  invisible(c(png = png_path, pdf = pdf_path))
}

sample_cluster_purity <- function(mat, metadata, k = length(STUDY_GROUP_LEVELS)) {
  if (ncol(mat) < k) return(tibble::tibble())
  hc <- stats::hclust(stats::dist(t(mat)))
  clusters <- stats::cutree(hc, k = k)
  as.data.frame(table(cluster = clusters, study_group = metadata[names(clusters), "study_group"])) |>
    dplyr::group_by(.data$cluster) |>
    dplyr::mutate(cluster_total = sum(.data$Freq), cluster_purity = max(.data$Freq) / .data$cluster_total) |>
    dplyr::ungroup()
}

plot_embedding_companion <- function(mat, metadata, method = c("pca", "umap")) {
  method <- match.arg(method)
  sample_mat <- t(mat)
  if (identical(method, "pca")) {
    emb <- stats::prcomp(sample_mat, center = FALSE, scale. = FALSE)$x[, 1:2, drop = FALSE]
    colnames(emb) <- c("Axis1", "Axis2")
    title <- "PCA on selected or fallback features"
  } else {
    if (!requireNamespace("uwot", quietly = TRUE)) return(NULL)
    set.seed(STABLR_SEED)
    emb <- uwot::umap(sample_mat, n_neighbors = min(10L, nrow(sample_mat) - 1L), metric = "euclidean")
    colnames(emb) <- c("Axis1", "Axis2")
    title <- "UMAP on selected or fallback features"
  }
  emb_df <- as.data.frame(emb) |>
    tibble::rownames_to_column("sample_id") |>
    dplyr::left_join(metadata |> tibble::rownames_to_column("sample_id"), by = "sample_id")
  ggplot2::ggplot(emb_df, ggplot2::aes(.data$Axis1, .data$Axis2, color = .data$study_group, shape = .data$protection)) +
    ggplot2::geom_point(size = 3, alpha = 0.9) +
    ggplot2::scale_color_manual(values = STUDY_GROUP_COLORS, drop = FALSE) +
    ggplot2::labs(title = title, x = NULL, y = NULL, color = "Study", shape = "P/NP") +
    ggplot2::theme_bw(base_size = 10) +
    ggplot2::theme(legend.position = "top")
}

plot_late_fusion_prediction_heatmap <- function(prediction_tbl, metadata) {
  prob_cols <- grep("^prob_", names(prediction_tbl), value = TRUE)
  if (length(prob_cols) == 0L) return(NULL)
  mat <- prediction_tbl |>
    dplyr::select(.data$sample_id, dplyr::all_of(prob_cols)) |>
    tibble::column_to_rownames("sample_id") |>
    as.matrix()
  mat <- t(mat)
  plot_df <- as.data.frame(mat, check.names = FALSE) |>
    tibble::rownames_to_column("class") |>
    tidyr::pivot_longer(cols = -dplyr::all_of("class"), names_to = "sample_id", values_to = "probability") |>
    dplyr::left_join(metadata |> tibble::rownames_to_column("sample_id"), by = "sample_id")
  ggplot2::ggplot(plot_df, ggplot2::aes(.data$sample_id, .data$class, fill = .data$probability)) +
    ggplot2::geom_tile(color = "white", linewidth = 0.15) +
    ggplot2::facet_grid(. ~ study_group, scales = "free_x", space = "free_x") +
    ggplot2::scale_fill_gradient(low = "white", high = "#2166AC", limits = c(0, 1)) +
    ggplot2::labs(title = "Late-fusion class probabilities", x = "Sample", y = NULL, fill = "Probability") +
    ggplot2::theme_bw(base_size = 9) +
    ggplot2::theme(axis.text.x = ggplot2::element_blank(), axis.ticks.x = ggplot2::element_blank())
}

plot_feature_overlap <- function(overlap_tbl) {
  overlap_tbl |>
    dplyr::mutate(feature = forcats::fct_reorder(.data$feature, as.integer(.data$selected), .fun = sum)) |>
    ggplot2::ggplot(ggplot2::aes(.data$source, .data$feature, fill = .data$selected)) +
    ggplot2::geom_tile(color = "white", linewidth = 0.2) +
    ggplot2::scale_fill_manual(values = c(`TRUE` = "#2166AC", `FALSE` = "grey90")) +
    ggplot2::labs(title = "Selected-feature overlap across analysis branches", x = NULL, y = NULL, fill = "Selected") +
    ggplot2::theme_bw(base_size = 9) +
    ggplot2::theme(axis.text.y = ggplot2::element_text(size = 6), legend.position = "top")
}

save_table <- function(x, filename, branch = infer_artifact_branch(filename)) {
  export_table(x, filename, branch = branch)
  invisible(x)
}

save_plot <- function(p, filename, width = 10, height = 7, branch = infer_artifact_branch(filename)) {
  export_plot(p, filename, branch = branch, width = width, height = height)
  invisible(p)
}


## CyTOF Single-View Sanity Check

This quick diagnostic fits STABL only on `cytof_celltype`. Readers should use this section to confirm the model runs, stratified bootstraps are active, and the stability path is sensible before interpreting the heavier all-view results. Any biological interpretation here is secondary to the all-view analysis below.


In [8]:
cytof_cache_candidates <- c(
  slurm_cytof = cached_rds_path("cytof", "stabl_fit_bundle"),
  notebook_cytof = cached_rds_path("cytof", "cytof_stabl_bundle"),
  slurm_single_view = cached_rds_path(file.path("single_view", "cytof_celltype"), "stabl_fit_bundle")
)
cytof_cache_path <- first_existing_path(cytof_cache_candidates)
cytof_stabl <- NULL

if (isTRUE(LOAD_CACHED_RESULTS) && !isTRUE(FORCE_RECOMPUTE) && !is.na(cytof_cache_path)) {
  cytof_stabl <- readRDS(cytof_cache_path)
  message("Loaded cached CyTOF STABL branch: ", cytof_cache_path)
} else if (isTRUE(RUN_CYTOF_SANITY_CHECK)) {
  x_cytof <- x_all_list$cytof_celltype
  y_cytof <- y_all
  metadata_cytof <- metadata_baseline
  cytof_bootstrap_strata <- data.frame(study_group = y_cytof, row.names = rownames(x_cytof))

  stopifnot(identical(rownames(x_cytof), names(y_cytof)))
  stopifnot(!anyNA(x_cytof))
  stopifnot(length(levels(droplevels(y_cytof))) == 3L)
  stopifnot(min(table(y_cytof)) >= 10L)

  cytof_stabl <- fit_stabl_elastic_net(
    x = x_cytof,
    y = y_cytof,
    seed = STABLR_SEED,
    view_name = "cytof_celltype",
    bootstrap_strata = cytof_bootstrap_strata
  )
  cache_object(cytof_stabl, "cytof_stabl_bundle", branch = "cytof")
} else {
  message("CyTOF STABL branch not loaded. Run `Rscript scratch/scripts/run_stablr_baseline_study_protection_branch.R cytof` or set RUN_CYTOF_SANITY_CHECK <- TRUE.")
}

if (!is.null(cytof_stabl)) {
  x_cytof <- x_all_list$cytof_celltype
  y_cytof <- y_all
  metadata_cytof <- metadata_baseline
  fit_cytof <- cytof_stabl$fit
  lambda_cytof <- cytof_stabl$lambda_grid
  importance_cytof <- importance_table(fit_cytof)
  selected_cytof <- get_feature_names_out(fit_cytof)
  cytof_signature_table <- immune_signature_table(
    fit = fit_cytof,
    x = x_cytof,
    y = y_cytof,
    lambda_grid = lambda_cytof,
    view = "cytof_celltype"
  )
  cytof_top_predictors_by_group <- cytof_signature_table |>
    dplyr::group_by(.data$study_group) |>
    dplyr::slice_max(order_by = .data$rank_score, n = TOP_FEATURES_PER_GROUP, with_ties = FALSE) |>
    dplyr::ungroup()

  save_table(importance_cytof, "cytof_celltype_importance.csv")
  save_table(cytof_signature_table, "cytof_celltype_signature_table.csv")

  selected_cytof
  head(importance_cytof, 20L)
  cytof_top_predictors_by_group
}


CyTOF STABL branch not loaded. Run `Rscript scratch/scripts/run_stablr_baseline_study_protection_branch.R cytof` or set RUN_CYTOF_SANITY_CHECK <- TRUE.



In [9]:
if (exists("fit_cytof") && exists("cytof_top_predictors_by_group")) {
  cytof_stability_path <- plot_stabl_path(
    fit_cytof,
    title = "CyTOF cell-type baseline study/protection elastic-net stability path"
  )
  cytof_top_predictor_plot <- plot_top_predictors_by_study_group(
    cytof_top_predictors_by_group,
    title = "Top CyTOF predictors by study group"
  )
  cytof_plot_choice <- choose_features_for_group_plot(fit_cytof, top_n = TOP_FALLBACK_FEATURES)
  cytof_feature_plot <- plot_features_by_study_class(
    features = cytof_plot_choice$features,
    x = x_cytof,
    metadata = metadata_cytof,
    fallback = cytof_plot_choice$fallback,
    title = "CyTOF feature distributions by study group"
  )
  cytof_heatmap <- plot_group_mean_heatmap(
    cytof_top_predictors_by_group,
    title = "CyTOF top-predictor group means"
  )

  save_plot(cytof_top_predictor_plot, "cytof_top_predictors_by_group.png", width = 10, height = 7)
  save_plot(cytof_feature_plot, "cytof_feature_distributions.png", width = 12, height = 8)
  save_plot(cytof_heatmap, "cytof_top_predictor_heatmap.png", width = 8, height = 7)

  cytof_stability_path
  cytof_top_predictor_plot
  cytof_feature_plot
  cytof_heatmap
}


## All Single-View STABL Fits

This section fits one multinomial STABL model per immune view using the same baseline samples. Interpret these results as view-specific evidence: a strong signal in one view means that assay block independently separates study/protection groups, while weak or unstable signals suggest that the block may need integration with other views or more samples.


In [10]:
single_view_cache_paths <- purrr::map_chr(names(x_all_list), function(view) {
  cached_rds_path(file.path("single_view", view), "stabl_fit_bundle")
}) |>
  stats::setNames(names(x_all_list))
single_view_aggregate_cache_path <- cached_rds_path("single_view", "single_view_stabl_results")
single_view_fits <- NULL

if (isTRUE(LOAD_CACHED_RESULTS) && !isTRUE(FORCE_RECOMPUTE) && all(file.exists(single_view_cache_paths))) {
  single_view_fits <- purrr::map(single_view_cache_paths, readRDS)
  message("Loaded cached single-view STABL branches: ", length(single_view_fits), " views")
} else if (isTRUE(LOAD_CACHED_RESULTS) && !isTRUE(FORCE_RECOMPUTE) && file.exists(single_view_aggregate_cache_path)) {
  single_view_cached <- readRDS(single_view_aggregate_cache_path)
  single_view_fits <- single_view_cached$fits
  message("Loaded cached aggregate single-view STABL results: ", single_view_aggregate_cache_path)
} else if (isTRUE(RUN_ALL_SINGLE_VIEW_STABL)) {
  single_view_fits <- purrr::imap(x_all_list, function(x, view) {
    seed_offset <- match(view, names(x_all_list)) * 100L
    fit_stabl_elastic_net(
      x = x,
      y = y_all,
      seed = STABLR_SEED + seed_offset,
      view_name = view,
      bootstrap_strata = bootstrap_strata_all
    )
  })
} else {
  missing_single_view <- names(single_view_cache_paths)[!file.exists(single_view_cache_paths)]
  message("Single-view branches not refit. Missing cached views: ", paste(missing_single_view, collapse = ", "))
}

if (!is.null(single_view_fits)) {
  single_view_importances <- purrr::imap_dfr(single_view_fits, function(result, view) {
    importance_table(result$fit) |>
      dplyr::mutate(view = view, .before = 1L)
  })

  single_view_selected <- purrr::imap_dfr(single_view_fits, function(result, view) {
    features <- get_feature_names_out(result$fit)
    tibble::tibble(view = view, feature = features, selected = TRUE)
  })

  single_view_signature_tables <- purrr::imap(single_view_fits, function(result, view) {
    immune_signature_table(
      fit = result$fit,
      x = x_all_list[[view]],
      y = y_all,
      lambda_grid = result$lambda_grid,
      view = view
    )
  })
  single_view_signatures <- dplyr::bind_rows(single_view_signature_tables)

  single_view_summary <- single_view_importances |>
    dplyr::group_by(.data$view) |>
    dplyr::summarise(
      n_features = dplyr::n(),
      n_selected = sum(.data$selected),
      max_stability = max(.data$stability_score, na.rm = TRUE),
      mean_top5_stability = mean(utils::head(sort(.data$stability_score, decreasing = TRUE), 5L)),
      .groups = "drop"
    ) |>
    dplyr::arrange(dplyr::desc(.data$max_stability))

  single_view_top_predictors <- single_view_signatures |>
    dplyr::group_by(.data$view, .data$study_group) |>
    dplyr::slice_max(order_by = .data$rank_score, n = TOP_FEATURES_PER_GROUP, with_ties = FALSE) |>
    dplyr::ungroup()

  single_view_summary_plot <- ggplot2::ggplot(
    single_view_summary,
    ggplot2::aes(x = forcats::fct_reorder(.data$view, .data$max_stability),
                 y = .data$max_stability, fill = .data$n_selected > 0)
  ) +
    ggplot2::geom_col(width = 0.72) +
    ggplot2::geom_text(ggplot2::aes(label = paste0("selected=", .data$n_selected)),
                       hjust = -0.05, size = 3) +
    ggplot2::coord_flip() +
    ggplot2::scale_y_continuous(limits = c(0, 1), expand = ggplot2::expansion(mult = c(0, 0.12))) +
    ggplot2::scale_fill_manual(values = c(`TRUE` = "#2166AC", `FALSE` = "grey70"), guide = "none") +
    ggplot2::labs(
      title = "Single-view STABL signal summary",
      subtitle = "Max stability shows the strongest feature in each assay block; selected counts reflect the FDP+ threshold.",
      x = NULL,
      y = "Maximum STABL stability"
    ) +
    ggplot2::theme_bw(base_size = 10) +
    ggplot2::theme(panel.grid.minor = ggplot2::element_blank())

  if (isTRUE(RUN_ALL_SINGLE_VIEW_STABL)) {
    cache_object(
      list(fits = single_view_fits, summary = single_view_summary,
           importances = single_view_importances, signatures = single_view_signatures,
           top_predictors = single_view_top_predictors),
      "single_view_stabl_results",
      branch = "single_view"
    )
  }

  save_table(single_view_summary, "single_view_summary.csv")
  save_table(single_view_importances, "single_view_importances.csv")
  save_table(single_view_signatures, "single_view_signature_table.csv")
  save_table(single_view_top_predictors, "single_view_top_predictors_by_group.csv")
  save_plot(single_view_summary_plot, "single_view_summary.png", width = 9, height = 6)

  single_view_summary
  single_view_selected
  single_view_top_predictors
  single_view_summary_plot
}


Single-view branches not refit. Missing cached views: cytof_celltype, exvivo_celltype, exvivo_enzyme, 6h_cyto_LPS, 6h_cyto_ssRNA40, 24h_cyto_iRBC, 24h_cyto_SEB, 24h_enzyme_iRBC, 24h_enzyme_SEB, 3d_cyto, 3d_enzyme



## All-View Early Fusion STABL

This section concatenates all preprocessed views after prefixing feature names with their view. Interpret this as the integrated multiomics model: selected or top-ranked features show which immune measurements remain informative when every assay block competes in the same STABL run.


In [11]:
x_all_early <- do.call(cbind, purrr::imap(x_all_list, prefix_feature_names))
stopifnot(identical(rownames(x_all_early), names(y_all)))

early_aggregate_cache_path <- cached_rds_path("early_fusion", "all_view_early_fusion_results")
early_branch_cache_path <- cached_rds_path("early_fusion", "stabl_fit_bundle")
all_early_stabl <- NULL
all_early_fitted_in_notebook <- FALSE

if (isTRUE(LOAD_CACHED_RESULTS) && !isTRUE(FORCE_RECOMPUTE) && file.exists(early_aggregate_cache_path)) {
  all_early_cached <- readRDS(early_aggregate_cache_path)
  all_early_stabl <- all_early_cached$fit_bundle %||% all_early_cached
  message("Loaded cached aggregate early-fusion results: ", early_aggregate_cache_path)
} else if (isTRUE(LOAD_CACHED_RESULTS) && !isTRUE(FORCE_RECOMPUTE) && file.exists(early_branch_cache_path)) {
  all_early_stabl <- readRDS(early_branch_cache_path)
  message("Loaded cached early-fusion STABL branch: ", early_branch_cache_path)
} else if (isTRUE(RUN_ALL_VIEW_EARLY_FUSION)) {
  all_early_stabl <- fit_stabl_elastic_net(
    x = x_all_early,
    y = y_all,
    seed = STABLR_SEED + 5000L,
    view_name = "all_view_early_fusion",
    bootstrap_strata = bootstrap_strata_all
  )
  all_early_fitted_in_notebook <- TRUE
} else {
  message("Early-fusion branch not loaded. Run `Rscript scratch/scripts/run_stablr_baseline_study_protection_branch.R early_fusion` or set RUN_ALL_VIEW_EARLY_FUSION <- TRUE.")
}

if (!is.null(all_early_stabl)) {
  fit_all_early <- all_early_stabl$fit
  lambda_all_early <- all_early_stabl$lambda_grid
  all_early_selected <- get_feature_names_out(fit_all_early)
  all_early_importance <- importance_table(fit_all_early)
  all_early_signature_raw <- immune_signature_table(
    fit = fit_all_early,
    x = x_all_early,
    y = y_all,
    lambda_grid = lambda_all_early,
    view = "all_view_early_fusion"
  )
  all_early_feature_parse <- split_prefixed_feature(unique(all_early_signature_raw$feature))
  all_early_signature_table <- all_early_signature_raw |>
    dplyr::left_join(all_early_feature_parse, by = "feature", suffix = c("", "_parsed")) |>
    dplyr::mutate(view = .data$view_parsed, feature_name = .data$feature_name) |>
    dplyr::select(-.data$view_parsed)

  all_early_top_predictors <- all_early_signature_table |>
    dplyr::group_by(.data$study_group) |>
    dplyr::slice_max(order_by = .data$rank_score, n = TOP_FEATURES_PER_GROUP, with_ties = FALSE) |>
    dplyr::ungroup()

  if (isTRUE(all_early_fitted_in_notebook)) {
    cache_object(
      list(fit_bundle = all_early_stabl, importance = all_early_importance,
           signature_table = all_early_signature_table, top_predictors = all_early_top_predictors,
           x = x_all_early, y = y_all),
      "all_view_early_fusion_results",
      branch = "early_fusion"
    )
  }

  save_table(all_early_importance, "all_view_early_fusion_importance.csv")
  save_table(all_early_signature_table, "all_view_early_fusion_signature_table.csv")
  save_table(all_early_top_predictors, "all_view_early_fusion_top_predictors_by_group.csv")

  all_early_selected
  head(all_early_importance, 30L)
  all_early_top_predictors
}


Early-fusion branch not loaded. Run `Rscript scratch/scripts/run_stablr_baseline_study_protection_branch.R early_fusion` or set RUN_ALL_VIEW_EARLY_FUSION <- TRUE.



In [12]:
if (exists("fit_all_early") && exists("all_early_top_predictors")) {
  all_early_stability_path <- plot_stabl_path(
    fit_all_early,
    title = "All-view early-fusion baseline study/protection elastic-net stability path"
  )
  all_early_top_predictor_plot <- plot_top_predictors_by_study_group(
    all_early_top_predictors,
    title = "Top all-view early-fusion predictors by study group"
  )
  all_early_heatmap <- plot_group_mean_heatmap(
    all_early_top_predictors,
    title = "All-view top-predictor group means"
  )
  all_early_view_contribution_plot <- plot_view_contribution(
    all_early_signature_table |>
      dplyr::group_by(.data$study_group) |>
      dplyr::slice_max(order_by = .data$rank_score, n = 25L, with_ties = FALSE) |>
      dplyr::ungroup(),
    title = "Assay-view contribution among top early-fusion predictors"
  )

  save_plot(all_early_top_predictor_plot, "all_view_top_predictors_by_group.png", width = 10, height = 7)
  save_plot(all_early_heatmap, "all_view_top_predictor_heatmap.png", width = 9, height = 8)
  save_plot(all_early_view_contribution_plot, "all_view_contribution.png", width = 9, height = 6)

  all_early_stability_path
  all_early_top_predictor_plot
  all_early_heatmap
  all_early_view_contribution_plot
}


## Direct One-Vs-Rest STABL Branches

This section runs or loads direct all-view early-fusion binomial STABL fits for each crossed label versus the rest. These branches are separate from cooperative OVR and are useful because the six-class target is strongly imbalanced, especially `TU_NP` with only 3 samples.

In [13]:
ovr_stabl_results <- purrr::set_names(vector("list", length(STUDY_GROUP_LEVELS)), STUDY_GROUP_LEVELS)
ovr_stabl_cache_paths <- purrr::map_chr(STUDY_GROUP_LEVELS, function(group) {
  cached_rds_path(file.path("ovr_stabl", group), "stabl_fit_bundle")
}) |>
  stats::setNames(STUDY_GROUP_LEVELS)

if (isTRUE(LOAD_CACHED_RESULTS) && !isTRUE(FORCE_RECOMPUTE)) {
  for (group in names(ovr_stabl_cache_paths)) {
    cached <- ovr_stabl_cache_paths[[group]]
    if (file.exists(cached)) {
      ovr_stabl_results[[group]] <- readRDS(cached)
      message("Loaded cached direct one-vs-rest STABL branch: ", cached)
    }
  }
}

missing_ovr_stabl_groups <- names(ovr_stabl_results)[vapply(ovr_stabl_results, is.null, logical(1L))]
if (length(missing_ovr_stabl_groups) > 0L && isTRUE(RUN_OVR_STABL)) {
  if (!exists("x_all_early")) {
    x_all_early <- do.call(cbind, purrr::imap(x_all_list, prefix_feature_names))
  }
  for (group in missing_ovr_stabl_groups) {
    y_bin <- factor(ifelse(as.character(y_all) == group, group, "rest"), levels = c("rest", group))
    names(y_bin) <- names(y_all)
    bootstrap_strata_ovr <- data.frame(outcome = y_bin, study_protection_group = y_all, row.names = names(y_all))
    fit <- fit_stabl_elastic_net(
      x = x_all_early,
      y = y_bin,
      seed = STABLR_SEED + 10000L + match(group, STUDY_GROUP_LEVELS) * 1000L,
      view_name = paste0("all_view_ovr_", group),
      bootstrap_strata = bootstrap_strata_ovr,
      family = "binomial"
    )
    branch <- file.path("ovr_stabl", group)
    cache_object(fit, "stabl_fit_bundle", branch = branch)
    ovr_stabl_results[[group]] <- fit
  }
} else if (length(missing_ovr_stabl_groups) > 0L) {
  message(
    "Direct OVR STABL branches not refit. Missing cached groups: ",
    paste(missing_ovr_stabl_groups, collapse = ", "),
    ". Run `Rscript scratch/scripts/run_stablr_baseline_study_protection_branch.R ovr_stabl:<group>` or set RUN_OVR_STABL <- TRUE."
  )
}

ovr_stabl_importance <- purrr::imap_dfr(ovr_stabl_results, function(bundle, group) {
  if (is.null(bundle) || is.null(bundle$fit)) return(tibble::tibble())
  importance_table(bundle$fit) |>
    dplyr::mutate(ovr_group = group, .before = 1L)
})

ovr_stabl_selected <- if (nrow(ovr_stabl_importance) > 0L && "selected" %in% names(ovr_stabl_importance)) {
  ovr_stabl_importance |>
    dplyr::filter(.data$selected) |>
    dplyr::select(.data$ovr_group, .data$feature, .data$stability_score)
} else {
  tibble::tibble(ovr_group = character(), feature = character(), stability_score = numeric())
}

if (nrow(ovr_stabl_importance) > 0L) save_table(ovr_stabl_importance, "ovr_stabl_importance.csv", branch = "ovr_stabl")
if (nrow(ovr_stabl_selected) > 0L) save_table(ovr_stabl_selected, "ovr_stabl_selected_features.csv", branch = "ovr_stabl")

ovr_stabl_selected
head(ovr_stabl_importance, 30L)


Direct OVR STABL branches not refit. Missing cached groups: EG_P, EG_NP, TU_P, TU_NP, GA_P, GA_NP. Run `Rscript scratch/scripts/run_stablr_baseline_study_protection_branch.R ovr_stabl:<group>` or set RUN_OVR_STABL <- TRUE.



ovr_group,feature,stability_score
<chr>,<chr>,<dbl>


<0 x 0 matrix>

## Multiclass Late Fusion STABL

This section fits or loads the true multiclass late-fusion branch. Each assay view first contributes class probabilities from its selected features, then a non-negative stacked weighting combines those probabilities for the primary `EG/GA/TU` endpoint. Interpret the weights as predictive contribution in this feasibility workflow; training-set metrics are descriptive until the publication nested-CV section is enabled.

In [14]:
late_fusion_cache_path <- cached_rds_path("late_fusion", "late_fusion_result")

if (isTRUE(LOAD_CACHED_RESULTS) && !isTRUE(FORCE_RECOMPUTE) && file.exists(late_fusion_cache_path)) {
  late_fusion_result <- readRDS(late_fusion_cache_path)
  message("Loaded cached late-fusion result: ", late_fusion_cache_path)
} else if (isTRUE(RUN_LATE_FUSION)) {
  late_fusion_result <- stabl_multiomic_train_validate(
    x_train_list = x_all_list,
    y_train = y_all,
    lambda_grid = "auto",
    base_learner = STABL_BASE_LEARNER,
    family = STABL_FAMILY,
    n_bootstraps = N_BOOTSTRAPS,
    artificial_type = ARTIFICIAL_TYPE,
    artificial_proportion = ARTIFICIAL_PROPORTION,
    sample_fraction = SAMPLE_FRACTION,
    replace = FALSE,
    stratify_bootstrap = TRUE,
    bootstrap_strata_train = bootstrap_strata_all,
    l1_ratio = ELASTIC_NET_L1_RATIO,
    random_state = STABLR_SEED + 7000L,
    n_lambda = N_LAMBDA,
    workers = N_WORKERS,
    fdr_threshold_range = FDR_THRESHOLD_RANGE,
    late_fusion = TRUE,
    n_iter_lf = N_ITER_LF
  )
  cache_object(late_fusion_result, "late_fusion_result", branch = "late_fusion")
} else {
  late_fusion_result <- NULL
  message("Late fusion not loaded. Run the SLURM late_fusion branch or set RUN_LATE_FUSION <- TRUE.")
}

if (!is.null(late_fusion_result) && !is.null(late_fusion_result$late_fusion)) {
  late_fusion_weights <- late_fusion_result$late_fusion$weights |>
    tibble::rownames_to_column("view") |>
    dplyr::arrange(dplyr::desc(.data$Associated_weight))
  late_fusion_predictions <- late_fusion_result$late_fusion$train_predictions |>
    tibble::rownames_to_column("sample_id") |>
    dplyr::mutate(truth = as.character(y_all[.data$sample_id]), .after = "sample_id")
  late_fusion_confusion <- confusion_table(late_fusion_predictions$truth, late_fusion_predictions$predicted_class)
  late_fusion_metrics <- classification_metric_table(late_fusion_predictions$truth, late_fusion_predictions$predicted_class) |>
    dplyr::bind_rows(tibble::tibble(metric = "log_loss", value = late_fusion_result$late_fusion$log_loss))
  late_fusion_weight_plot <- ggplot2::ggplot(
    late_fusion_weights,
    ggplot2::aes(x = forcats::fct_reorder(.data$view, .data$Associated_weight), y = .data$Associated_weight)
  ) +
    ggplot2::geom_col(fill = "#2166AC") +
    ggplot2::coord_flip() +
    ggplot2::labs(title = "Late-fusion assay-view weights", x = NULL, y = "Stacked probability weight") +
    ggplot2::theme_bw(base_size = 10)
  late_fusion_prediction_heatmap <- plot_late_fusion_prediction_heatmap(late_fusion_predictions, metadata_baseline)

  save_table(late_fusion_weights, "late_fusion_weights.csv")
  save_table(late_fusion_predictions, "late_fusion_train_predictions.csv")
  save_table(late_fusion_confusion, "late_fusion_confusion_matrix.csv")
  late_fusion_per_class_metrics <- per_class_metric_table(late_fusion_predictions$truth, late_fusion_predictions$predicted_class)

  save_table(late_fusion_metrics, "late_fusion_metrics.csv")
  save_table(late_fusion_per_class_metrics, "late_fusion_per_class_metrics.csv")
  save_plot(late_fusion_weight_plot, "late_fusion_view_weights.png", width = 8, height = 5)
  if (!is.null(late_fusion_prediction_heatmap)) {
    save_plot(late_fusion_prediction_heatmap, "late_fusion_prediction_heatmap.png", width = 10, height = 4)
  }

  late_fusion_weights
  late_fusion_metrics
  late_fusion_per_class_metrics
  late_fusion_confusion
}


Late fusion not loaded. Run the SLURM late_fusion branch or set RUN_LATE_FUSION <- TRUE.



## Cooperative One-Vs-Rest Branches

This auxiliary section runs or loads cooperative learning for six crossed-label one-vs-rest branches. It is intentionally binomial one-vs-rest because the current cooperative backend is not the primary multinomial model. Use these outputs as a cross-view sensitivity check: repeated features across cooperative and STABL branches are stronger candidates for follow-up.

In [15]:
cooperative_results <- purrr::set_names(vector("list", length(STUDY_GROUP_LEVELS)), STUDY_GROUP_LEVELS)
cooperative_cache_paths <- purrr::map_chr(STUDY_GROUP_LEVELS, function(group) {
  cached_rds_path(file.path("cooperative_ovr", group), "cooperative_fit")
}) |>
  stats::setNames(STUDY_GROUP_LEVELS)

if (isTRUE(LOAD_CACHED_RESULTS) && !isTRUE(FORCE_RECOMPUTE)) {
  for (group in names(cooperative_cache_paths)) {
    cached <- cooperative_cache_paths[[group]]
    if (file.exists(cached)) {
      cooperative_results[[group]] <- readRDS(cached)
      message("Loaded cached cooperative one-vs-rest branch: ", cached)
    }
  }
}

missing_cooperative_groups <- names(cooperative_results)[vapply(cooperative_results, is.null, logical(1L))]
if (length(missing_cooperative_groups) > 0L && isTRUE(RUN_COOPERATIVE_OVR)) {
  if (!requireNamespace("multiview", quietly = TRUE)) {
    stop("Package 'multiview' is required for cooperative one-vs-rest branches.", call. = FALSE)
  }
  for (group in missing_cooperative_groups) {
    y_bin <- factor(ifelse(as.character(y_all) == group, group, "rest"), levels = c("rest", group))
    names(y_bin) <- names(y_all)
    bootstrap_strata_ovr <- data.frame(outcome = y_bin, study_protection_group = y_all, row.names = names(y_all))
    fit <- stabl_multiomic_train_validate(
      x_train_list = x_all_list,
      y_train = y_bin,
      lambda_grid = "auto",
      base_learner = STABL_BASE_LEARNER,
      family = "binomial",
      n_bootstraps = N_BOOTSTRAPS,
      artificial_type = ARTIFICIAL_TYPE,
      artificial_proportion = ARTIFICIAL_PROPORTION,
      sample_fraction = SAMPLE_FRACTION,
      replace = FALSE,
      stratify_bootstrap = TRUE,
      bootstrap_strata_train = bootstrap_strata_ovr,
      l1_ratio = ELASTIC_NET_L1_RATIO,
      random_state = STABLR_SEED + match(group, STUDY_GROUP_LEVELS) * 1000L,
      n_lambda = N_LAMBDA,
      workers = N_WORKERS,
      fdr_threshold_range = FDR_THRESHOLD_RANGE,
      cooperative_fusion = TRUE,
      rho = COOPERATIVE_RHO,
      cooperation_selection = "cv",
      cooperation_selector = "lambda.1se",
      cooperation_type_measure = "deviance",
      cooperation_nfolds = 3L
    )
    branch <- file.path("cooperative_ovr", group)
    cache_object(fit, "cooperative_fit", branch = branch)
    cooperative_results[[group]] <- fit
  }
} else if (length(missing_cooperative_groups) > 0L) {
  message("Cooperative branches not refit. Missing cached groups: ", paste(missing_cooperative_groups, collapse = ", "))
}

cooperative_diagnostics <- purrr::imap_dfr(cooperative_results, function(fit, group) {
  if (is.null(fit) || is.null(fit$cooperative_fusion)) return(tibble::tibble())
  get_cooperative_diagnostics(fit) |>
    dplyr::mutate(ovr_group = group, .before = 1L)
})

cooperative_features <- purrr::imap_dfr(cooperative_results, function(fit, group) {
  if (is.null(fit) || is.null(fit$cooperative_fusion)) return(tibble::tibble())
  get_cooperative_features(fit) |>
    purrr::imap_dfr(function(features, view) tibble::tibble(ovr_group = group, view = view, feature = features))
})

if (nrow(cooperative_diagnostics) > 0L) save_table(cooperative_diagnostics, "cooperative_ovr_diagnostics.csv", branch = "cooperative_ovr")
if (nrow(cooperative_features) > 0L) save_table(cooperative_features, "cooperative_ovr_features.csv", branch = "cooperative_ovr")
cooperative_diagnostics
cooperative_features


Cooperative branches not refit. Missing cached groups: EG_P, EG_NP, TU_P, TU_NP, GA_P, GA_NP



<0 x 0 matrix>

<0 x 0 matrix>

## Top Study-Group Predictors

This section turns the STABL results into an interpretation table. A feature ranks highly when it has both high stability and a large class-specific elastic-net beta. The beta sign points toward or away from a study/protection group in the multinomial model; stability tells how consistently the feature survived resampling and artificial-feature competition.


In [16]:
if (exists("all_early_top_predictors")) {
  top_study_group_predictors <- all_early_top_predictors |>
    dplyr::arrange(.data$study_group, dplyr::desc(.data$rank_score)) |>
    dplyr::mutate(
      interpretation = dplyr::case_when(
        .data$beta > 0 & .data$one_vs_rest_mean_diff > 0 ~ "higher in this group and positive model beta",
        .data$beta > 0 & .data$one_vs_rest_mean_diff <= 0 ~ "positive model beta but group mean is not elevated",
        .data$beta < 0 & .data$one_vs_rest_mean_diff < 0 ~ "lower in this group and negative model beta",
        .data$beta < 0 & .data$one_vs_rest_mean_diff >= 0 ~ "negative model beta but group mean is not depleted",
        TRUE ~ "near-zero beta"
      )
    )

  save_table(top_study_group_predictors, "top_study_group_predictors_interpreted.csv")
  top_study_group_predictors
}


## Feature-Level Immune Signature Plots

This section visualizes the top predictors as measured values, not only model scores. Boxplots and jittered samples show whether candidate features are elevated or depleted in each study/protection group; P/NP point shapes help check whether the signal is broadly consistent or potentially driven by protection-status imbalance.


In [17]:
if (exists("fit_all_early") && exists("all_early_top_predictors") && exists("x_all_early")) {
  all_early_plot_choice <- choose_features_for_group_plot(fit_all_early, top_n = TOP_FALLBACK_FEATURES)
  all_early_features_to_plot <- unique(c(all_early_top_predictors$feature, all_early_plot_choice$features))
  all_early_features_to_plot <- utils::head(all_early_features_to_plot, 18L)

  all_early_feature_distribution_plot <- plot_features_by_study_class(
    features = all_early_features_to_plot,
    x = x_all_early,
    metadata = metadata_baseline,
    fallback = all_early_plot_choice$fallback,
    title = "All-view top immune signature features by study group"
  )

  save_plot(all_early_feature_distribution_plot, "all_view_feature_distributions.png", width = 14, height = 10)
  all_early_feature_distribution_plot
}


## Selected-Feature Clustering And Embedding Diagnostics

These plots ask whether the selected or fallback immune-signature features visibly separate samples by `EG_P`, `EG_NP`, `TU_P`, `TU_NP`, `GA_P`, and `GA_NP`. Heatmaps prioritize feature-level patterns and cluster structure; PCA/UMAP companion plots show whether the same signatures produce low-dimensional group separation. Treat P/NP as an annotation only, useful for spotting imbalance inside clusters.

In [18]:
visualize_heatmap_cache_path <- cached_rds_path("visualize", "selected_feature_heatmap_matrix")

if (isTRUE(LOAD_CACHED_RESULTS) && !isTRUE(FORCE_RECOMPUTE) &&
    file.exists(visualize_heatmap_cache_path) && !isTRUE(RUN_CLUSTERING_VISUALIZATIONS)) {
  selected_feature_heatmap_cache <- readRDS(visualize_heatmap_cache_path)
  selected_feature_matrix <- selected_feature_heatmap_cache$matrix
  heatmap_feature_tbl <- selected_feature_heatmap_cache$feature_metadata
  selected_cluster_purity <- load_cached_table("visualize", "selected_feature_sample_cluster_purity")
  selected_feature_pca_scores <- load_cached_table("visualize", "selected_feature_pca_scores")
  cached_visualization_figures <- cached_artifact_paths("visualize", "figures")
  message("Loaded cached visualization matrix: ", visualize_heatmap_cache_path)
  list(
    selected_feature_matrix_dim = dim(selected_feature_matrix),
    selected_feature_count = nrow(heatmap_feature_tbl),
    cached_figures = cached_visualization_figures
  )
} else if (isTRUE(RUN_CLUSTERING_VISUALIZATIONS) && exists("fit_all_early") && exists("x_all_early")) {
  heatmap_feature_tbl <- choose_features_for_heatmap(fit_all_early, top_n = TOP_HEATMAP_FEATURES)
  heatmap_feature_tbl <- heatmap_feature_tbl |>
    dplyr::left_join(split_prefixed_feature(heatmap_feature_tbl$feature), by = "feature") |>
    dplyr::left_join(
      all_early_signature_table |>
        dplyr::group_by(.data$feature) |>
        dplyr::slice_max(order_by = .data$rank_score, n = 1L, with_ties = FALSE) |>
        dplyr::ungroup() |>
        dplyr::select(.data$feature, strongest_group = .data$study_group, beta, abs_beta, rank_score),
      by = "feature"
    ) |>
    dplyr::arrange(dplyr::desc(.data$selected), dplyr::desc(.data$stability_score), .data$view, .data$feature_name)

  selected_feature_matrix <- make_feature_heatmap_matrix(x_all_early, heatmap_feature_tbl$feature)
  heatmap_feature_tbl <- heatmap_feature_tbl[match(rownames(selected_feature_matrix), heatmap_feature_tbl$feature), , drop = FALSE]
  cache_object(list(matrix = selected_feature_matrix, feature_metadata = heatmap_feature_tbl),
               "selected_feature_heatmap_matrix", branch = "early_fusion")
  save_table(heatmap_feature_tbl, "selected_feature_heatmap_features.csv")

  export_selected_feature_heatmap(
    selected_feature_matrix,
    sample_metadata = metadata_baseline,
    feature_metadata = heatmap_feature_tbl,
    filename = "all_view_selected_feature_heatmap.png",
    branch = "early_fusion",
    row_split = heatmap_feature_tbl$view,
    width = 13,
    height = 10
  )

  stability_weighted_tbl <- heatmap_feature_tbl |>
    dplyr::mutate(stability_beta_score = .data$stability_score * tidyr::replace_na(.data$abs_beta, 0)) |>
    dplyr::arrange(dplyr::desc(.data$stability_beta_score), .data$view, .data$feature_name)
  export_selected_feature_heatmap(
    selected_feature_matrix[stability_weighted_tbl$feature, , drop = FALSE],
    sample_metadata = metadata_baseline,
    feature_metadata = stability_weighted_tbl,
    filename = "stability_weighted_selected_feature_heatmap.png",
    branch = "early_fusion",
    row_split = stability_weighted_tbl$view,
    width = 13,
    height = 10
  )

  top_group_features <- all_early_top_predictors |>
    dplyr::distinct(.data$feature, .keep_all = TRUE) |>
    dplyr::left_join(choose_features_for_heatmap(fit_all_early, top_n = TOP_HEATMAP_FEATURES), by = "feature") |>
    dplyr::mutate(selected = tidyr::replace_na(.data$selected, FALSE), exploratory_fallback = tidyr::replace_na(.data$exploratory_fallback, TRUE))
  top_group_matrix <- make_feature_heatmap_matrix(x_all_early, top_group_features$feature)
  export_selected_feature_heatmap(
    top_group_matrix,
    sample_metadata = metadata_baseline,
    feature_metadata = top_group_features,
    filename = "per_study_group_top_feature_heatmap.png",
    branch = "early_fusion",
    row_split = top_group_features$study_group[match(rownames(top_group_matrix), top_group_features$feature)],
    width = 12,
    height = 8
  )

  selected_cluster_purity <- sample_cluster_purity(selected_feature_matrix, metadata_baseline)
  save_table(selected_cluster_purity, "selected_feature_sample_cluster_purity.csv")

  selected_feature_pca <- plot_embedding_companion(selected_feature_matrix, metadata_baseline, method = "pca")
  save_plot(selected_feature_pca, "selected_feature_pca.png", width = 7, height = 6)
  selected_feature_umap <- plot_embedding_companion(selected_feature_matrix, metadata_baseline, method = "umap")
  if (!is.null(selected_feature_umap)) save_plot(selected_feature_umap, "selected_feature_umap.png", width = 7, height = 6)

  overlap_sources <- list(
    early_fusion = all_early_selected,
    single_view = if (exists("single_view_selected")) paste(single_view_selected$view, single_view_selected$feature, sep = "__") else character()
  )
  if (exists("cooperative_features") && nrow(cooperative_features) > 0L) {
    overlap_sources$cooperative_ovr <- paste(cooperative_features$view, cooperative_features$feature, sep = "__")
  }
  overlap_universe <- unique(c(unlist(overlap_sources), heatmap_feature_tbl$feature))
  feature_overlap_tbl <- purrr::imap_dfr(overlap_sources, function(features, source) {
    tibble::tibble(source = source, feature = overlap_universe, selected = overlap_universe %in% features)
  })
  save_table(feature_overlap_tbl, "feature_overlap_branches.csv")
  feature_overlap_plot <- plot_feature_overlap(feature_overlap_tbl)
  save_plot(feature_overlap_plot, "feature_overlap_branches.png", width = 8, height = 10)

  selected_cluster_purity
  selected_feature_pca
  selected_feature_umap
  feature_overlap_plot
} else {
  message("Clustering visualizations not regenerated. Use cached figures under `scratch/outputs/stablr_baseline_study_protection_test/visualize/` or set RUN_CLUSTERING_VISUALIZATIONS <- TRUE.")
}


Clustering visualizations not regenerated. Use cached figures under `scratch/outputs/stablr_baseline_study_protection_test/visualize/` or set RUN_CLUSTERING_VISUALIZATIONS <- TRUE.



## Cross-View Interpretation

This section summarizes which assay views contribute most to the group-discriminating signatures. Treat this as a map of where the immune signal appears strongest across modalities and stimulation conditions, not as proof that one assay is biologically causal.


In [19]:
if (exists("single_view_summary") && exists("all_early_signature_table") && exists("top_study_group_predictors")) {
  cross_view_summary <- all_early_signature_table |>
    dplyr::group_by(.data$study_group, .data$view) |>
    dplyr::summarise(
      n_top25_features = sum(.data$rank_score >= sort(.data$rank_score, decreasing = TRUE)[min(25L, dplyr::n())], na.rm = TRUE),
      n_selected = sum(.data$selected, na.rm = TRUE),
      max_stability = max(.data$stability_score, na.rm = TRUE),
      total_rank_score = sum(.data$rank_score, na.rm = TRUE),
      strongest_feature = .data$feature_name[which.max(.data$rank_score)],
      .groups = "drop"
    ) |>
    dplyr::arrange(.data$study_group, dplyr::desc(.data$total_rank_score))

  immune_signature_narrative <- top_study_group_predictors |>
    dplyr::group_by(.data$study_group) |>
    dplyr::summarise(
      top_features = paste(.data$feature_name, collapse = "; "),
      dominant_views = paste(unique(.data$view), collapse = "; "),
      interpretation_notes = paste(.data$interpretation, collapse = "; "),
      .groups = "drop"
    )

  save_table(cross_view_summary, "cross_view_summary.csv")
  save_table(immune_signature_narrative, "immune_signature_narrative.csv")

  cross_view_summary
  immune_signature_narrative
}


## Publication-Scale Validation

This optional section is disabled by default. Before making biological claims, rerun the workflow with more bootstraps, a broader lambda/alpha grid, nested cross-validation, and sensitivity analyses such as alternative artificial-feature schemes. The expected output is validation evidence for model performance and feature robustness, not new exploratory feature lists.


In [20]:
nested_cv_cache_path <- cached_rds_path("nested_cv", "nested_cv_result")

if (isTRUE(LOAD_CACHED_RESULTS) && !isTRUE(FORCE_RECOMPUTE) && file.exists(nested_cv_cache_path)) {
  nested_cv_result <- readRDS(nested_cv_cache_path)
  message("Loaded cached nested-CV result: ", nested_cv_cache_path)
} else if (isTRUE(RUN_PUBLICATION_NESTED_CV)) {
  nested_cv_result <- stabl_multiomic_nested_cv(
    x_list = x_all_list,
    y = y_all,
    candidates = NULL,  # Package default: each view plus one all-view early-fusion candidate.
    lambda_grid = "auto",
    outer_v = 3L,
    outer_repeats = 1L,
    inner_v = 2L,
    stratified = TRUE,
    strata = y_all,
    metric = "ber",
    family = STABL_FAMILY,
    base_learner = STABL_BASE_LEARNER,
    n_bootstraps = N_BOOTSTRAPS,
    artificial_type = ARTIFICIAL_TYPE,
    sample_fraction = SAMPLE_FRACTION,
    replace = FALSE,
    stratify_bootstrap = TRUE,

    bootstrap_strata = data.frame(study_protection_group = y_all, row.names = names(y_all)),
    random_state = STABLR_SEED + 9000L,
    n_lambda = N_LAMBDA,
    workers = N_WORKERS,
    cv_workers = 1L
  )

  cache_object(nested_cv_result, "nested_cv_result", branch = "nested_cv")
} else {
  nested_cv_result <- NULL
  message("Nested CV not loaded. Run the SLURM nested_cv branch or set RUN_PUBLICATION_NESTED_CV <- TRUE.")
}

nested_cv_result


Nested CV not loaded. Run the SLURM nested_cv branch or set RUN_PUBLICATION_NESTED_CV <- TRUE.



NULL

## Interpretation Summary And Caveats

Use this section to write the final reader-facing takeaways after executing the notebook. Summaries should name the strongest candidate immune signatures for `EG_P`, `EG_NP`, `TU_P`, `TU_NP`, `GA_P`, and `GA_NP`, identify the assay views carrying those signals, and clearly state whether each claim is STABL-selected, top-ranked fallback, or only descriptive.

Current caveats: the cohort is small, the study-protection-group target is imbalanced, P/NP is not modeled, and the default bootstrap/lambda settings are feasibility-scale. Treat these outputs as candidate immune signatures until publication-scale validation is complete.
